[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/09_maximum_likelihood_and_map_estimation/exercises.ipynb)

# Exercises — Module 09 — Maximum Likelihood and MAP Estimation

20 fully solved problems in four tiers: L0 Concept Checks (4), L1 Foundations (6), L2 Applications in AI/ML and Physics (6), L3 Challenge Proofs (4).

Every problem carries a **Statement**, an **Intuition**, a step-by-step **Solution**, a boxed answer, a **Key takeaway**, and a code cell that recomputes the answer. Numbered results such as Theorem 4.5 or Proof 5.6 refer to [`first_principles.ipynb`](first_principles.ipynb).

The preamble below is shared by every solution cell in this notebook. All randomness is drawn from the single generator `rng`, so every printed number is reproducible.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

from scipy import optimize, stats
from scipy.special import expit, logsumexp

print("ready")

ready


## L0 — Concept Checks

### Problem L0.1 — Likelihood Is Not a Probability Density in the Parameter

**Statement**

For $X \sim \mathcal{N}(\theta, 1)$ observed once at $x = 3$, write the likelihood $L(\theta)$ and compute $\int_{-\infty}^{\infty} L(\theta)\,d\theta$. What does the result show, and when does a probability statement about $\theta$ become available?

**Intuition**

Normalization is a statement about the *data* argument of $p(x\mid\theta)$, and nothing forces it to hold in the parameter argument. Integrating $L$ over $\theta$ in two different models — once finite, once infinite — settles the question.

**Solution**

The likelihood is the sampling density read as a function of $\theta$:

$$
L(\theta) = p(x = 3 \mid \theta) = \frac{1}{\sqrt{2\pi}}\exp\left(-\frac{(3-\theta)^2}{2}\right) .
$$

**Integrating over $\theta$.** By symmetry of $(3-\theta)^2$ in $\theta$ about $3$, this is a Gaussian kernel in $\theta$ with the same normalizing constant, so

$$
\int_{-\infty}^{\infty} L(\theta)\,d\theta = 1 .
$$

Here the integral happens to be $1$ — a coincidence of the location-family structure, *not* a general fact. For $X\sim\mathcal{N}(0,\theta)$ observed at $x$, $L(\theta) = (2\pi\theta)^{-1/2}e^{-x^2/(2\theta)}$ and $\int_0^\infty L(\theta)d\theta = \infty$ (the integrand decays only like $\theta^{-1/2}$). So likelihood is not a density in $\theta$ in general; it lacks the normalization property entirely.

**What is guaranteed.** $\int p(x\mid\theta)\,dx = 1$ for each fixed $\theta$ — normalization holds in the *data* argument, which is the only one probability theory constrains.

**When probability over $\theta$ appears.** Only after supplying a prior $p(\theta)$ and normalizing:

$$
p(\theta\mid x) = \frac{L(\theta)p(\theta)}{\int L(\theta')p(\theta')d\theta'} .
$$

$$
\boxed{L(\theta) = p(x\mid\theta) \text{ normalizes in } x, \text{ not in } \theta; \text{ probabilities over } \theta \text{ require a prior}}
$$

*Key takeaway:* "Likelihood of $\theta$" is a ranking, not a probability; the missing ingredient is always the prior.

The cell below recomputes every number claimed in Problem L0.1.

In [2]:
from scipy import integrate

# Case 1: N(theta, 1) observed at x = 3 -- the likelihood happens to integrate to 1 in theta.
lik_loc = lambda th: stats.norm.pdf(3.0, loc=th, scale=1.0)
val_loc, _ = integrate.quad(lik_loc, -80, 80)
print(f"location family : integral of L over theta = {val_loc:.12f}  (hand: 1)")
assert np.isclose(val_loc, 1.0)

# Case 2: N(0, theta) observed at x = 2 -- the same construction diverges.
lik_scale = lambda th: stats.norm.pdf(2.0, loc=0.0, scale=np.sqrt(th))
print("\nscale family    : partial integrals of L over theta in (0, U)")
for U in (1e2, 1e4, 1e6, 1e8):
    v, _ = integrate.quad(lik_scale, 1e-8, U, limit=400)
    print(f"   U = {U:9.0e}   integral = {v:12.4f}   2*sqrt(U/(2*pi)) = {2*np.sqrt(U/(2*np.pi)):12.4f}")
v_small, _ = integrate.quad(lik_scale, 1e-8, 1e4, limit=400)
v_big, _ = integrate.quad(lik_scale, 1e-8, 1e8, limit=400)
assert v_big > 50 * v_small          # grows like sqrt(U): unbounded, so no normalization exists

location family : integral of L over theta = 1.000000000000  (hand: 1)

scale family    : partial integrals of L over theta in (0, U)
   U =     1e+02   integral =       6.1379   2*sqrt(U/(2*pi)) =       7.9788
   U =     1e+04   integral =      77.8044   2*sqrt(U/(2*pi)) =      79.7885
   U =     1e+06   integral =     795.8862   2*sqrt(U/(2*pi)) =     797.8846
   U =     1e+08   integral =    7978.8458   2*sqrt(U/(2*pi)) =    7978.8456


### Problem L0.2 — Three Heads Out of Three

**Statement**

A coin is flipped 3 times and lands heads every time. Compute the MLE of $p$, explain why it is unreasonable, and compute the MAP estimate under a $\text{Beta}(2,2)$ prior.

**Intuition**

With no counterweight the likelihood will happily place all its mass on the observed outcomes, so the fix has to come from outside the data. A prior supplies exactly one pseudo-success and one pseudo-failure here.

**Solution**

**MLE.** With $k = 3$, $n = 3$, $\ell(p) = 3\ln p$, which is strictly increasing on $(0,1]$, so the maximum is at the boundary:

$$
\hat{p}_{\text{MLE}} = \frac{k}{n} = 1 .
$$

**Why it is unreasonable.** This asserts $P(\text{tails}) = 0$: a single future tail would be an event the model calls impossible, giving infinite loss ($-\ln 0 = \infty$) in any log-likelihood evaluation. Three observations cannot rule out a whole outcome, yet the MLE does exactly that because it optimizes fit with no counterweight. The same failure is why unsmoothed $n$-gram models assign zero probability — hence infinite perplexity — to any unseen token sequence.

**MAP with $\text{Beta}(2,2)$.** The prior density is $p(p) \propto p^{\alpha-1}(1-p)^{\beta-1} = p(1-p)$, so

$$
\ln p(p\mid x) = (k + \alpha - 1)\ln p + (n-k+\beta-1)\ln(1-p) + \text{const} = 4\ln p + 1\cdot\ln(1-p) + \text{const} .
$$

Differentiating: $\frac{4}{p} - \frac{1}{1-p} = 0 \Rightarrow 4(1-p) = p \Rightarrow p = 4/5$. In general,

$$
\hat p_{\text{MAP}} = \frac{k+\alpha-1}{n+\alpha+\beta-2} = \frac{3+1}{3+2} = 0.8 .
$$

The prior contributed one pseudo-head and one pseudo-tail, so the model now allows tails with probability $0.2$. (For comparison, the posterior *mean* is $\frac{k+\alpha}{n+\alpha+\beta} = \frac{5}{7} \approx 0.714$, more conservative still.)

$$
\boxed{\hat p_{\text{MLE}} = 1 \ (\text{degenerate}), \qquad \hat p_{\text{MAP}} = \tfrac{4}{5} = 0.8}
$$

*Key takeaway:* Priors act as pseudo-counts and are the cheapest fix for the MLE's willingness to declare unseen outcomes impossible.

The cell below recomputes every number claimed in Problem L0.2.

In [3]:
k, n = 3, 3
alpha, beta_ = 2.0, 2.0
p_mle = k / n
p_map = (k + alpha - 1) / (n + alpha + beta_ - 2)
p_mean = (k + alpha) / (n + alpha + beta_)

# recompute the MAP by direct maximization of the log-posterior, not by the formula
grid = np.linspace(1e-9, 1 - 1e-9, 2_000_001)
logpost = (k + alpha - 1) * np.log(grid) + (n - k + beta_ - 1) * np.log1p(-grid)
p_map_grid = grid[np.argmax(logpost)]

print(f"MLE            = {p_mle:.6f}")
print(f"MAP (formula)  = {p_map:.6f}   MAP (grid search) = {p_map_grid:.6f}")
print(f"posterior mean = {p_mean:.6f}  (Beta(5,2) mean = {stats.beta.mean(k+alpha, n-k+beta_):.6f})")
assert np.isclose(p_map, 0.8) and np.isclose(p_map_grid, 0.8, atol=1e-5)
assert np.isclose(p_mean, 5 / 7)

MLE            = 1.000000
MAP (formula)  = 0.800000   MAP (grid search) = 0.800000
posterior mean = 0.714286  (Beta(5,2) mean = 0.714286)


### Problem L0.3 — Is the MLE Unbiased?

**Statement**

Show that the Gaussian variance MLE $\hat\sigma^2 = \frac1n\sum_i(x_i-\bar{x})^2$ is biased, quantify the bias, and explain why this does not disqualify maximum likelihood.

**Intuition**

Fitting the mean uses up one degree of freedom, so squared deviations measured around $\bar x$ are systematically smaller than deviations around $\mu$. The deficit is exactly one observation's worth.

**Solution**

**Bias computation.** Write $\sum_i (x_i-\bar x)^2 = \sum_i (x_i-\mu)^2 - n(\bar x - \mu)^2$ (expand and use $\sum_i(x_i - \mu) = n(\bar x - \mu)$). Taking expectations with $E[(x_i-\mu)^2] = \sigma^2$ and $E[(\bar x-\mu)^2] = \sigma^2/n$:

$$
E\left[\sum_i (x_i-\bar x)^2\right] = n\sigma^2 - n\cdot\frac{\sigma^2}{n} = (n-1)\sigma^2 .
$$

Therefore

$$
E\left[\hat\sigma^2\right] = \frac{n-1}{n}\sigma^2, \qquad \text{Bias} = -\frac{\sigma^2}{n} .
$$

The unbiased alternative divides by $n-1$: $S^2 = \frac{1}{n-1}\sum_i (x_i-\bar x)^2$. The intuitive reason for the deficit is that $\bar{x}$ is itself fitted to the data and sits closer to the sample than $\mu$ does, so squared deviations around it are systematically too small — one degree of freedom is consumed.

**Why the bias is acceptable.** Three points:

1. It is $O(1/n)$ and vanishes: the MLE is asymptotically unbiased and consistent.
2. Bias is not the target. In mean squared error the MLE actually *beats* $S^2$ here: $\text{MSE}(\hat\sigma^2) = \frac{2n-1}{n^2}\sigma^4 \lt \frac{2}{n-1}\sigma^4 = \text{MSE}(S^2)$.
3. Unbiasedness cannot survive the invariance property. Even though $S^2$ is unbiased for $\sigma^2$, $\sqrt{S^2}$ is *not* unbiased for $\sigma$ (Jensen: $E[\sqrt{S^2}] \lt \sigma$). Any estimator that is unbiased on one scale is biased on another, so no estimation principle can be unbiased for all parameterizations.

$$
\boxed{E[\hat\sigma^2_{\text{MLE}}] = \tfrac{n-1}{n}\sigma^2; \text{ biased, consistent, and lower MSE than } S^2}
$$

*Key takeaway:* Maximum likelihood trades exact unbiasedness for invariance and asymptotic efficiency — usually a good trade.

The cell below recomputes every number claimed in Problem L0.3.

In [4]:
n, sigma = 8, 2.0
reps = 400_000
x = rng.normal(0.0, sigma, size=(reps, n))
sig2_mle = x.var(axis=1, ddof=0)
s2_unb = x.var(axis=1, ddof=1)

print(f"E[sigma^2_MLE] = {sig2_mle.mean():.5f}   predicted (n-1)/n * sigma^2 = {(n-1)/n*sigma**2:.5f}")
print(f"E[S^2]         = {s2_unb.mean():.5f}   predicted sigma^2            = {sigma**2:.5f}")
mse_mle = np.mean((sig2_mle - sigma**2) ** 2)
mse_unb = np.mean((s2_unb - sigma**2) ** 2)
print(f"MSE(MLE) = {mse_mle:.5f}  predicted (2n-1)/n^2 * sigma^4 = {(2*n-1)/n**2*sigma**4:.5f}")
print(f"MSE(S^2) = {mse_unb:.5f}  predicted 2/(n-1) * sigma^4    = {2/(n-1)*sigma**4:.5f}")
assert np.isclose(sig2_mle.mean(), (n - 1) / n * sigma**2, rtol=0.02)
assert mse_mle < mse_unb

E[sigma^2_MLE] = 3.49863   predicted (n-1)/n * sigma^2 = 3.50000
E[S^2]         = 3.99844   predicted sigma^2            = 4.00000


MSE(MLE) = 3.76161  predicted (2n-1)/n^2 * sigma^4 = 3.75000
MSE(S^2) = 4.58481  predicted 2/(n-1) * sigma^4    = 4.57143


### Problem L0.4 — MLE, MAP, and Posterior Mean — Three Different Answers

**Statement**

For $k = 2$ successes in $n = 10$ trials with a $\text{Beta}(3,1)$ prior, compute the MLE, the MAP estimate, and the posterior mean. Explain what each one optimizes.

**Intuition**

The three numbers differ because they answer three different questions: which $\theta$ predicts the data best, which $\theta$ is most probable after seeing them, and which $\theta$ minimizes expected squared error.

**Solution**

**Posterior.** For a Binomial likelihood with a $\text{Beta}(\alpha,\beta)$ prior, the posterior is $\text{Beta}(k+\alpha,\ n-k+\beta) = \text{Beta}(5, 9)$.

**MLE.** Maximizes $p(x\mid\theta)$:

$$
\hat p_{\text{MLE}} = \frac{k}{n} = \frac{2}{10} = 0.200 .
$$

**MAP.** Maximizes $p(\theta\mid x)$, i.e. the posterior mode $\frac{a-1}{a+b-2}$ for $\text{Beta}(a,b)$ with $a,b \gt 1$:

$$
\hat p_{\text{MAP}} = \frac{5-1}{5+9-2} = \frac{4}{12} = 0.333 .
$$

**Posterior mean.** Minimizes expected squared error:

$$
E[p\mid x] = \frac{a}{a+b} = \frac{5}{14} \approx 0.357 .
$$

**What each optimizes.**

| Estimator | Objective | Decision-theoretic loss |
|---|---|---|
| MLE | $\max_\theta p(x \mid \theta)$ | none (no prior; a frequentist criterion) |
| MAP | $\max_\theta p(\theta \mid x)$ | $0$–$1$ loss (limit of an all-or-nothing penalty) |
| Posterior mean | $E[\theta \mid x]$ | squared error $(\hat\theta - \theta)^2$ |
| Posterior median | $F^{-1}_{\theta \mid x}(0.5)$ | absolute error $\vert \hat\theta - \theta\vert$ |

The three disagree here because the $\text{Beta}(3,1)$ prior is informative and pushes upward (it favors large $p$), while $n = 10$ is small. With $n = 1000$ and $k = 200$ all three would be within $0.003$ of $0.2$.

$$
\boxed{\hat p_{\text{MLE}} = 0.200, \quad \hat p_{\text{MAP}} = 0.333, \quad E[p\mid x] \approx 0.357}
$$

*Key takeaway:* The estimator you report is a choice of loss function; naming it makes the choice explicit instead of accidental.

The cell below recomputes every number claimed in Problem L0.4.

In [5]:
k, n, alpha, beta_ = 2, 10, 3.0, 1.0
a, b = k + alpha, n - k + beta_                  # posterior Beta(a, b) = Beta(5, 9)
p_mle = k / n
p_map = (a - 1) / (a + b - 2)
p_mean = a / (a + b)
p_med = stats.beta.ppf(0.5, a, b)
print(f"posterior = Beta({a:.0f}, {b:.0f})")
print(f"MLE            = {p_mle:.4f}   (hand 0.200)")
print(f"MAP            = {p_map:.4f}   (hand 0.333)")
print(f"posterior mean = {p_mean:.4f}   (hand 5/14 = {5/14:.4f})")
print(f"posterior med  = {p_med:.4f}")
assert np.isclose(p_map, 1 / 3) and np.isclose(p_mean, 5 / 14)
grid = np.linspace(1e-6, 1 - 1e-6, 200_001)       # mode found by search, not by formula
p_map_grid = grid[np.argmax(stats.beta.logpdf(grid, a, b))]
print(f"MAP by grid search = {p_map_grid:.6f}")
assert np.isclose(p_map_grid, 1 / 3, atol=1e-4)

k2, n2 = 200, 1000                                # the same three estimators at n = 1000
a2, b2 = k2 + alpha, n2 - k2 + beta_
print(f"\nn = 1000: MLE {k2/n2:.4f}, MAP {(a2-1)/(a2+b2-2):.4f}, mean {a2/(a2+b2):.4f}")
assert max(abs(k2/n2 - (a2-1)/(a2+b2-2)), abs(k2/n2 - a2/(a2+b2))) < 0.003

posterior = Beta(5, 9)
MLE            = 0.2000   (hand 0.200)
MAP            = 0.3333   (hand 0.333)
posterior mean = 0.3571   (hand 5/14 = 0.3571)
posterior med  = 0.3502
MAP by grid search = 0.333335

n = 1000: MLE 0.2000, MAP 0.2016, mean 0.2022


## L1 — Foundations

### Problem L1.1 — MLE, Fisher Information, and a Confidence Interval for the Exponential Rate

**Statement**

Let $X_1,\ldots,X_n$ be i.i.d. $\text{Exponential}(\lambda)$ with density $f(x) = \lambda e^{-\lambda x}$, $x \gt 0$. Derive $\hat\lambda_{\text{MLE}}$, the Fisher information $I(\lambda)$, and a $95\%$ confidence interval. Check whether $\hat\lambda$ is unbiased.

**Intuition**

The exponential is an exponential family with $\bar x$ as its sufficient statistic, so the MLE is the invariance image of $\bar x$ under $\mu \mapsto 1/\mu$. Because $1/\bar x$ is a convex function of $\bar x$, Jensen already predicts an upward bias.

**Solution**

**Log-likelihood and MLE.**

$$
\ell(\lambda) = n\ln\lambda - \lambda\sum_{i=1}^n x_i, \qquad \ell'(\lambda) = \frac{n}{\lambda} - \sum_i x_i = 0 \quad\Longrightarrow\quad \hat\lambda = \frac{n}{\sum_i x_i} = \frac{1}{\bar{x}} .
$$

Since $\ell''(\lambda) = -n/\lambda^2 \lt 0$, this is the unique maximum. Note it is the invariance property in action: $\hat\lambda = 1/\hat\mu$ where $\hat\mu = \bar{x}$ is the MLE of the mean $1/\lambda$.

**Fisher information.** For a single observation, $\ln f = \ln\lambda - \lambda x$, so $s = \frac{1}{\lambda} - x$ and $\partial_\lambda^2 \ln f = -1/\lambda^2$. Hence

$$
I(\lambda) = -E\left[-\frac{1}{\lambda^2}\right] = \frac{1}{\lambda^2} .
$$

Cross-check via the score variance: $\operatorname{Var}(1/\lambda - X) = \operatorname{Var}(X) = 1/\lambda^2$ ✓.

**Asymptotic distribution and interval.** $\sqrt{n}(\hat\lambda - \lambda)\xrightarrow{d}\mathcal{N}(0,\lambda^2)$, so

$$
\hat\lambda \pm 1.96\,\frac{\hat\lambda}{\sqrt{n}} = \hat\lambda\left(1 \pm \frac{1.96}{\sqrt n}\right) .
$$

For $n = 100$ and $\hat\lambda = 2.0$ this is $2.0 \pm 0.392 = (1.61, 2.39)$. Because the interval is multiplicative, a log-scale interval $\exp\left(\ln\hat\lambda \pm 1.96/\sqrt n\right) = (1.64, 2.43)$ is better behaved and never crosses zero.

**Bias.** $\sum_i X_i \sim \text{Gamma}(n,\lambda)$ and $E\left[1/G\right] = \frac{\lambda}{n-1}$ for $G\sim\text{Gamma}(n,\lambda)$, so

$$
E[\hat\lambda] = nE\left[\frac{1}{\sum_i X_i}\right] = \frac{n\lambda}{n-1} \gt \lambda .
$$

The MLE overestimates by a factor $\frac{n}{n-1}$; the bias-corrected estimator is $\frac{n-1}{n}\hat\lambda$.

$$
\boxed{\hat\lambda = 1/\bar x, \quad I(\lambda) = 1/\lambda^2, \quad \text{CI} = \hat\lambda\left(1 \pm 1.96/\sqrt n\right), \quad E[\hat\lambda] = \tfrac{n}{n-1}\lambda}
$$

*Key takeaway:* Fisher information $1/\lambda^2$ makes the standard error proportional to $\lambda$ — rate parameters are estimated with constant *relative* precision.

The cell below recomputes every number claimed in Problem L1.1.

In [6]:
lam0, n = 2.0, 100
z = stats.norm.ppf(0.975)
lam_hat = 2.0                                    # the illustrative value used in the solution
wald = lam_hat * np.array([1 - z / np.sqrt(n), 1 + z / np.sqrt(n)])
logci = np.exp(np.log(lam_hat) + np.array([-1, 1]) * z / np.sqrt(n))
print(f"z_0.975 = {z:.6f}")
print(f"Wald CI      = ({wald[0]:.4f}, {wald[1]:.4f})   (hand 1.61, 2.39)")
print(f"log-scale CI = ({logci[0]:.4f}, {logci[1]:.4f})   (hand 1.64, 2.43)")
assert np.allclose(wald, [1.6080, 2.3920], atol=1e-3)
assert np.allclose(logci, [1.6440, 2.4331], atol=1e-3)

for nn in (5, 20, 100):                          # E[lambda-hat] = n/(n-1) * lambda
    xs = rng.exponential(1 / lam0, size=(400_000, nn))
    lh = 1 / xs.mean(axis=1)
    print(f"n = {nn:3d}: E[lambda-hat] = {lh.mean():.5f}   predicted {nn/(nn-1)*lam0:.5f}")
    assert np.isclose(lh.mean(), nn / (nn - 1) * lam0, rtol=0.02)

xs = rng.exponential(1 / lam0, size=(20_000, n))
lh = 1 / xs.mean(axis=1)
lo, hi = lh * (1 - z / np.sqrt(n)), lh * (1 + z / np.sqrt(n))
print(f"\nWald coverage at n = {n}: {np.mean((lo < lam0) & (lam0 < hi)):.4f}  (nominal 0.95)")

z_0.975 = 1.959964
Wald CI      = (1.6080, 2.3920)   (hand 1.61, 2.39)
log-scale CI = (1.6440, 2.4330)   (hand 1.64, 2.43)
n =   5: E[lambda-hat] = 2.50249   predicted 2.50000


n =  20: E[lambda-hat] = 2.10485   predicted 2.10526


n = 100: E[lambda-hat] = 2.02000   predicted 2.02020

Wald coverage at n = 100: 0.9518  (nominal 0.95)


### Problem L1.2 — A Non-Regular Model — $\text{Unif}(0,\theta)$

**Statement**

Let $X_1,\ldots,X_n \sim \text{Unif}(0,\theta)$. Find the MLE, its exact distribution, its bias, and the unbiased corrected version. Explain why the standard asymptotic theory does not apply.

**Intuition**

The likelihood here is a cliff, not a hill: it is flat-then-zero, so no derivative ever vanishes and every conclusion drawn from the score equation is void. The estimator turns out to be *better* than the regular theory allows.

**Solution**

**MLE.** The joint density is

$$
L(\theta) = \prod_{i=1}^n \frac{1}{\theta}\mathbf{1}\{0 \le x_i \le \theta\} = \theta^{-n}\,\mathbf{1}\left\{\theta \ge x_{(n)}\right\}, \qquad x_{(n)} = \max_i x_i .
$$

On its support $L$ is strictly decreasing in $\theta$, so no interior stationary point exists; the maximum is at the smallest admissible value:

$$
\hat\theta = x_{(n)} .
$$

**Exact distribution.** $P(X_{(n)} \le t) = \prod_i P(X_i \le t) = (t/\theta)^n$ for $0 \le t \le\theta$, so $f_{X_{(n)}}(t) = nt^{n-1}/\theta^n$ and

$$
E\left[X_{(n)}\right] = \int_0^\theta t\cdot\frac{nt^{n-1}}{\theta^n}dt = \frac{n}{n+1}\theta .
$$

**Bias and correction.** Bias $= -\theta/(n+1)$, systematically low because a maximum of $n$ samples can never exceed $\theta$. The corrected estimator

$$
\tilde\theta = \frac{n+1}{n}X_{(n)}
$$

is unbiased. It is in fact the UMVUE, and the module now supplies the machinery: $X_{(n)}$ is sufficient by the factorization $L(\theta) = \theta^{-n}\mathbf{1}\{x_{(n)} \le \theta\}$ (Theorem 4.8, with $h \equiv 1$), and it is **complete**, because if $E_\theta\left[g(X_{(n)})\right] = 0$ for every $\theta \gt 0$ then

$$
\int_0^\theta g(t)\,\frac{n t^{n-1}}{\theta^n}\,dt = 0 \quad \Longrightarrow \quad \int_0^\theta g(t)\,t^{n-1}\,dt = 0 \quad \text{for all } \theta \gt 0 ,
$$

and differentiating in $\theta$ gives $g(\theta)\theta^{n-1} = 0$, hence $g \equiv 0$. Theorem 4.10 (Lehmann–Scheffé) then makes $\tilde\theta$ the unique UMVUE.

**Why asymptotics fail.** Regularity requires the support to be free of $\theta$; here it is not, and differentiating under the integral sign is illegal. The consequences are visible:

$$
n\left(\theta - X_{(n)}\right) \xrightarrow{d} \text{Exponential}(1/\theta) ,
$$

since $P\left(n(\theta - X_{(n)}) \gt s\right) = \left(1 - \frac{s}{n\theta}\right)^n \to e^{-s/\theta}$. The convergence rate is $n^{-1}$, not $n^{-1/2}$; the limit law is exponential, not normal; and the estimator is *superefficient*, beating the Cramér–Rao bound — which is no contradiction because the bound's hypotheses are violated.

$$
\boxed{\hat\theta = x_{(n)}, \quad E[\hat\theta] = \tfrac{n}{n+1}\theta, \quad \tilde\theta = \tfrac{n+1}{n}x_{(n)}, \quad \text{rate } n^{-1}}
$$

*Key takeaway:* When the support depends on the parameter, calculus and the asymptotic normal theory both break — solve the optimization directly.

The cell below recomputes every number claimed in Problem L1.2.

In [7]:
theta0, n = 3.0, 40
reps = 200_000
xmax = rng.uniform(0, theta0, size=(reps, n)).max(axis=1)

print(f"E[X_(n)]  = {xmax.mean():.5f}   predicted n/(n+1)*theta = {n/(n+1)*theta0:.5f}")
corrected = (n + 1) / n * xmax
print(f"E[tilde]  = {corrected.mean():.5f}   predicted theta         = {theta0:.5f}")
assert np.isclose(xmax.mean(), n / (n + 1) * theta0, rtol=0.01)
assert np.isclose(corrected.mean(), theta0, rtol=0.01)

# n(theta - X_(n)) is Exponential(mean theta): compare quantiles, not just the mean
w = n * (theta0 - xmax)
qs = np.array([0.1, 0.25, 0.5, 0.75, 0.9])
print("\nquantiles of n(theta - X_(n)) vs Exponential(mean = theta):")
for q, emp in zip(qs, np.quantile(w, qs)):
    print(f"   q = {q:4.2f}   empirical {emp:7.4f}   exponential {stats.expon.ppf(q, scale=theta0):7.4f}")
assert np.allclose(np.quantile(w, qs), stats.expon.ppf(qs, scale=theta0), rtol=0.05)

# the corrected estimator beats the (inapplicable) Cramer-Rao bound 1/(n I) = theta^2/n
print(f"\nVar(tilde theta) = {corrected.var():.6f}   theta^2/n = {theta0**2/n:.6f}"
      f"   ratio = {corrected.var()/(theta0**2/n):.4f}")
assert corrected.var() < theta0**2 / n

E[X_(n)]  = 2.92659   predicted n/(n+1)*theta = 2.92683
E[tilde]  = 2.99975   predicted theta         = 3.00000

quantiles of n(theta - X_(n)) vs Exponential(mean = theta):
   q = 0.10   empirical  0.3157   exponential  0.3161
   q = 0.25   empirical  0.8602   exponential  0.8630
   q = 0.50   empirical  2.0637   exponential  2.0794
   q = 0.75   empirical  4.1023   exponential  4.1589
   q = 0.90   empirical  6.7411   exponential  6.9078

Var(tilde theta) = 0.005414   theta^2/n = 0.225000   ratio = 0.0241


### Problem L1.3 — Poisson Rate — Information, Efficiency, and Counting Error

**Statement**

For $X_1,\ldots,X_n \sim \text{Poisson}(\lambda)$, derive the MLE, the Fisher information, the Cramér–Rao bound, and show that the MLE is exactly efficient at every $n$.

**Intuition**

Poisson is the textbook case where the score is exactly affine in the estimator, which is the equality condition in Cauchy–Schwarz. Efficiency should therefore hold at every $n$, not just asymptotically.

**Solution**

**MLE.** With $p(x\mid\lambda) = e^{-\lambda}\lambda^x/x!$,

$$
\ell(\lambda) = -n\lambda + \left(\sum_i x_i\right)\ln\lambda - \sum_i \ln(x_i!), \qquad \ell'(\lambda) = -n + \frac{\sum_i x_i}{\lambda} = 0 \Longrightarrow \hat\lambda = \bar{x} .
$$

**Fisher information.** $\partial_\lambda \ln p = -1 + x/\lambda$, so $\partial^2_\lambda \ln p = -x/\lambda^2$ and

$$
I(\lambda) = -E\left[-\frac{X}{\lambda^2}\right] = \frac{E[X]}{\lambda^2} = \frac{\lambda}{\lambda^2} = \frac{1}{\lambda} .
$$

Equivalently $\operatorname{Var}(s) = \operatorname{Var}(X)/\lambda^2 = \lambda/\lambda^2 = 1/\lambda$ ✓.

**Cramér–Rao bound and efficiency.**

$$
\operatorname{Var}(T) \ \ge\ \frac{1}{nI(\lambda)} = \frac{\lambda}{n} .
$$

The MLE is unbiased ($E[\bar X] = \lambda$) with $\operatorname{Var}(\bar X) = \lambda/n$, attaining the bound *exactly for every $n$*, not merely asymptotically. The equality condition of Proof 5.4 confirms this: the total score is

$$
s_n(\lambda) = -n + \frac{\sum_i X_i}{\lambda} = \frac{n}{\lambda}\left(\bar{X} - \lambda\right) ,
$$

an affine function of $\bar X - \lambda$, exactly the Cauchy–Schwarz equality case.

**Counting error in physics.** For a single measurement of $N$ counts ($n = 1$), $\hat\lambda = N$ with standard error $\sqrt{\lambda}\approx\sqrt{N}$ — the ubiquitous "$\sqrt{N}$ error bar". The *relative* error $1/\sqrt{N}$ is why detecting a $1\%$ effect requires $\sim 10^4$ counts and a $0.1\%$ effect requires $\sim 10^6$.

$$
\boxed{\hat\lambda = \bar{x}, \quad I(\lambda) = 1/\lambda, \quad \operatorname{Var}(\hat\lambda) = \lambda/n = \text{CRLB}}
$$

*Key takeaway:* In exponential families the natural sufficient statistic is exactly efficient — the Cauchy–Schwarz equality case is not an accident but a structural feature.

The cell below recomputes every number claimed in Problem L1.3.

In [8]:
lam0, n = 3.0, 25
reps = 300_000
x = rng.poisson(lam0, size=(reps, n))
lam_hat = x.mean(axis=1)

I_lam = 1 / lam0
crlb = 1 / (n * I_lam)
print(f"I(lambda) = {I_lam:.6f},  CRLB = 1/(nI) = {crlb:.6f}")
print(f"E[lambda-hat] = {lam_hat.mean():.5f} (unbiased, true {lam0})")
print(f"Var(lambda-hat) = {lam_hat.var(ddof=1):.6f}   predicted lambda/n = {lam0/n:.6f}")
assert np.isclose(lam_hat.var(ddof=1), lam0 / n, rtol=0.02)

# equality condition of Cauchy-Schwarz: the total score is affine in lambda-hat
s_n = -n + x.sum(axis=1) / lam0
print(f"max |s_n - (n/lambda)(lambda-hat - lambda)| = "
      f"{np.abs(s_n - (n / lam0) * (lam_hat - lam0)).max():.3e}")
assert np.abs(s_n - (n / lam0) * (lam_hat - lam0)).max() < 1e-9

for N in (100, 10_000, 1_000_000):               # the sqrt(N) counting error
    print(f"N = {N:9d} counts: se = sqrt(N) = {np.sqrt(N):10.2f}, relative = {1/np.sqrt(N):.5f}")

I(lambda) = 0.333333,  CRLB = 1/(nI) = 0.120000
E[lambda-hat] = 3.00048 (unbiased, true 3.0)
Var(lambda-hat) = 0.120142   predicted lambda/n = 0.120000
max |s_n - (n/lambda)(lambda-hat - lambda)| = 5.329e-15
N =       100 counts: se = sqrt(N) =      10.00, relative = 0.10000
N =     10000 counts: se = sqrt(N) =     100.00, relative = 0.01000
N =   1000000 counts: se = sqrt(N) =    1000.00, relative = 0.00100


### Problem L1.4 — The Invariance Property in Practice

**Statement**

Given $\hat\lambda = \bar{x}$ for a Poisson sample, find the MLEs of (a) $P(X = 0) = e^{-\lambda}$, (b) $\ln\lambda$, and (c) the odds $\frac{p}{1-p}$ for a Bernoulli sample. Then show that invariance and unbiasedness are incompatible.

**Intuition**

Maximum likelihood is a statement about which point of $\Theta$ maximizes a function, and that point pushes forward through any map. Unbiasedness is a statement about an integral, and integrals do not commute with nonlinear maps.

**Solution**

**Invariance (Theorem 4.1).** If $\hat\theta$ maximizes $L$, then for any $g$ the induced profile likelihood $L^*(\eta) = \sup\{L(\theta) : g(\theta)=\eta\}$ is maximized at $\eta = g(\hat\theta)$. So MLEs simply push forward through functions.

**(a)** $\widehat{P(X=0)} = e^{-\hat\lambda} = e^{-\bar x}$.

**(b)** $\widehat{\ln\lambda} = \ln\bar{x}$.

**(c)** $\widehat{\text{odds}} = \frac{\hat p}{1-\hat p} = \frac{\bar{x}}{1-\bar{x}}$, which is the maximum likelihood logit-scale estimate used in logistic regression.

**Incompatibility with unbiasedness.** Consider (a) with $n = 1$, $X\sim\text{Poisson}(\lambda)$. The MLE is $e^{-X}$, whose expectation is

$$
E\left[e^{-X}\right] = \sum_{x=0}^\infty e^{-x}\frac{e^{-\lambda}\lambda^x}{x!} = e^{-\lambda}\sum_x \frac{(\lambda/e)^x}{x!} = e^{-\lambda}e^{\lambda/e} = e^{-\lambda(1 - e^{-1})} \ne e^{-\lambda} .
$$

So the MLE of $e^{-\lambda}$ is biased. Meanwhile the unique unbiased estimator of $e^{-\lambda}$ at $n = 1$ is $T(X) = \mathbf{1}\{X = 0\}$, since $E[T] = P(X=0) = e^{-\lambda}$; by completeness of the Poisson family it is the only one. It reports either $0$ or $1$ as the estimate of a probability strictly between them — unbiased and useless. The same construction applied to $e^{-2\lambda}$ is worse still: $E\left[(-1)^X\right] = e^{\lambda(-1-1)} = e^{-2\lambda}$, so the unique unbiased estimator of a positive quantity is negative half the time.

**The general principle.** Unbiasedness is defined relative to a parameterization: $E[T] = g(\theta)$ cannot hold simultaneously for $g$ and for a nonlinear $h\circ g$, because Jensen's inequality gives $E[h(T)] \ne h(E[T])$ unless $h$ is affine. Maximum likelihood chooses invariance; unbiasedness chooses one privileged scale.

$$
\boxed{\widehat{g(\theta)} = g(\hat\theta) \text{ always; unbiasedness holds on at most one scale}}
$$

*Key takeaway:* Invariance is the property that makes MLE usable across reparameterizations — and it is precisely what rules out exact unbiasedness.

The cell below recomputes every number claimed in Problem L1.4.

In [9]:
lam0 = 1.3
xs = np.arange(0, 200)
pmf = stats.poisson.pmf(xs, lam0)

E_exp_negX = pmf @ np.exp(-xs)
print(f"E[e^-X]        = {E_exp_negX:.12f}   predicted e^(-lambda(1-1/e)) = "
      f"{np.exp(-lam0 * (1 - np.exp(-1))):.12f}")
print(f"target e^-lambda = {np.exp(-lam0):.12f}  -> MLE of e^-lambda is biased upward")
assert np.isclose(E_exp_negX, np.exp(-lam0 * (1 - np.exp(-1))))
assert E_exp_negX > np.exp(-lam0)

E_indicator = pmf @ (xs == 0)
E_alternating = pmf @ ((-1.0) ** xs)
print(f"\nE[1{{X = 0}}]    = {E_indicator:.12f}  = e^-lambda: unbiased, but takes only 0 or 1")
print(f"E[(-1)^X]      = {E_alternating:.12f}  = e^(-2 lambda) = {np.exp(-2*lam0):.12f}")
assert np.isclose(E_indicator, np.exp(-lam0)) and np.isclose(E_alternating, np.exp(-2 * lam0))

n = 40                                            # MLEs push forward through g
x = rng.poisson(lam0, size=n)
lam_hat = x.mean()
print(f"\nlambda-hat = {lam_hat:.4f}: P(X=0)-hat = {np.exp(-lam_hat):.6f}, "
      f"ln(lambda)-hat = {np.log(lam_hat):.6f}")

E[e^-X]        = 0.439658615765   predicted e^(-lambda(1-1/e)) = 0.439658615765
target e^-lambda = 0.272531793034  -> MLE of e^-lambda is biased upward

E[1{X = 0}]    = 0.272531793034  = e^-lambda: unbiased, but takes only 0 or 1
E[(-1)^X]      = 0.074273578214  = e^(-2 lambda) = 0.074273578214

lambda-hat = 1.1000: P(X=0)-hat = 0.332871, ln(lambda)-hat = 0.095310


### Problem L1.5 — Multivariate Gaussian — MLE of Mean and Covariance

**Statement**

For $x_1,\ldots,x_n \in \mathbb{R}^d$ i.i.d. $\mathcal{N}(\mu,\Sigma)$ with $\Sigma \succ 0$, derive $\hat\mu$ and $\hat\Sigma$. Discuss what happens when $n \lt d$.

**Intuition**

The covariance MLE is the scatter matrix, and a scatter matrix built from $n$ centred points cannot have rank above $n-1$. Everything about the $n \lt d$ regime follows from that rank count.

**Solution**

**Log-likelihood.** Using $\ln\det\Sigma^{-1} = -\ln\det\Sigma$,

$$
\ell(\mu,\Sigma) = -\frac{nd}{2}\ln(2\pi) - \frac{n}{2}\ln\det\Sigma - \frac12\sum_{i=1}^n (x_i-\mu)^{\top}\Sigma^{-1}(x_i-\mu) .
$$

**Mean.** Differentiating the quadratic form in $\mu$:

$$
\nabla_\mu \ell = \Sigma^{-1}\sum_{i=1}^n (x_i - \mu) = 0 \quad\Longrightarrow\quad \hat\mu = \bar{x} = \frac1n\sum_i x_i ,
$$

using invertibility of $\Sigma^{-1}$.

**Covariance.** Write the quadratic term with the trace trick, $a^{\top}Ma = \operatorname{tr}(Maa^{\top})$, and define the scatter matrix $S = \frac1n\sum_i (x_i-\bar x)(x_i-\bar x)^{\top}$. Then, with $\Lambda = \Sigma^{-1}$,

$$
\ell = \frac{n}{2}\ln\det\Lambda - \frac{n}{2}\operatorname{tr}(\Lambda S) + \text{const} .
$$

Using the matrix identities $\nabla_\Lambda \ln\det\Lambda = \Lambda^{-1}$ and $\nabla_\Lambda\operatorname{tr}(\Lambda S) = S$:

$$
\nabla_\Lambda \ell = \frac{n}{2}\left(\Lambda^{-1} - S\right) = 0 \quad\Longrightarrow\quad \hat\Sigma = S = \frac1n\sum_{i=1}^n (x_i-\bar x)(x_i-\bar x)^{\top} .
$$

(Concavity of $\ln\det$ on the positive-definite cone confirms this is the global maximum.) As in the scalar case the divisor is $n$, so $E[\hat\Sigma] = \frac{n-1}{n}\Sigma$.

**The $n \lt d$ regime.** $S$ is a sum of $n$ rank-one matrices minus the mean adjustment, so $\operatorname{rank}(S) \le n-1 \lt d$: the MLE is *singular*, $\det\hat\Sigma = 0$, and $\hat\Sigma^{-1}$ does not exist. The likelihood is in fact unbounded — one can send probability mass to infinity along a direction with no data. Remedies, all of which are MAP estimation with an inverse-Wishart prior in disguise:

$$
\hat\Sigma_{\text{shrunk}} = (1-\gamma)S + \gamma\,\frac{\operatorname{tr}(S)}{d}I \quad \text{(Ledoit–Wolf)}, \qquad \hat\Sigma_{\text{ridge}} = S + \epsilon I .
$$

$$
\boxed{\hat\mu = \bar{x}, \qquad \hat\Sigma = \frac1n\sum_i (x_i-\bar x)(x_i-\bar x)^{\top}, \quad \text{singular when } n \le d}
$$

*Key takeaway:* Covariance estimation needs $n \gg d$; below that threshold regularization is not optional but required for the estimate to exist.

The cell below recomputes every number claimed in Problem L1.5.

In [10]:
d, n = 3, 500
A = rng.normal(size=(d, d))
Sigma_true = A @ A.T + d * np.eye(d)
mu_true = np.array([1.0, -2.0, 0.5])
X = rng.multivariate_normal(mu_true, Sigma_true, size=n)

mu_hat = X.mean(axis=0)
Xc = X - mu_hat
Sigma_hat = Xc.T @ Xc / n
print(f"mu-hat = {mu_hat}")
print(f"max |Sigma-hat - np.cov(bias=True)| = {np.abs(Sigma_hat - np.cov(X.T, bias=True)).max():.3e}")
assert np.allclose(Sigma_hat, np.cov(X.T, bias=True))

# the MLE maximizes the Gaussian log-likelihood: perturbing it must lower the value
def gauss_ll(mu, S):
    return stats.multivariate_normal(mu, S).logpdf(X).sum()

base = gauss_ll(mu_hat, Sigma_hat)
worse = [gauss_ll(mu_hat, Sigma_hat * c) for c in (0.9, 1.1)]
print(f"loglik at MLE = {base:.4f}, at 0.9*Sigma = {worse[0]:.4f}, at 1.1*Sigma = {worse[1]:.4f}")
assert base > max(worse)

n_small = 2                                       # n < d: the scatter matrix is singular
Xs = rng.multivariate_normal(mu_true, Sigma_true, size=n_small)
Ss = (Xs - Xs.mean(axis=0)).T @ (Xs - Xs.mean(axis=0)) / n_small
print(f"\nn = {n_small} < d = {d}: rank(S) = {np.linalg.matrix_rank(Ss)} <= n-1 = {n_small-1}, "
      f"det(S) = {np.linalg.det(Ss):.3e}")
assert np.linalg.matrix_rank(Ss) <= n_small - 1

mu-hat = [ 0.8351 -1.9633  0.3989]
max |Sigma-hat - np.cov(bias=True)| = 0.000e+00
loglik at MLE = -3204.0164, at 0.9*Sigma = -3208.3294, at 1.1*Sigma = -3207.3172

n = 2 < d = 3: rank(S) = 1 <= n-1 = 1, det(S) = -7.466e-35


### Problem L1.6 — Beta-Prior MAP and Its Convergence to the MLE

**Statement**

For $k$ successes in $n$ Bernoulli trials with a $\text{Beta}(\alpha,\beta)$ prior, derive the MAP estimate, express it as a weighted average of the MLE and the prior mode, and quantify the rate at which the prior's influence decays.

**Intuition**

A conjugate prior enters the log-posterior as extra counts, so the MAP is the MLE of an enlarged, fictitious dataset. Reading off the weights of that mixture gives the $O(1/n)$ decay immediately.

**Solution**

**Objective.** Up to constants,

$$
\ln p(p\mid x) = \underbrace{k\ln p + (n-k)\ln(1-p)}_{\ell(p)} + \underbrace{(\alpha-1)\ln p + (\beta-1)\ln(1-p)}_{\ln \text{prior}} .
$$

Collecting terms, this is the log-likelihood of a Bernoulli experiment with $k + \alpha - 1$ successes out of $n + \alpha + \beta - 2$ trials, so by Derivation 5.1(a),

$$
\hat p_{\text{MAP}} = \frac{k+\alpha-1}{n+\alpha+\beta-2} \qquad (\alpha, \beta \gt 1) .
$$

**Weighted-average form.** Let $m = \alpha+\beta-2$ be the prior's pseudo-count and $p_0 = \frac{\alpha-1}{\alpha+\beta-2}$ the prior mode. Then

$$
\hat p_{\text{MAP}} = \frac{n}{n+m}\cdot\underbrace{\frac{k}{n}}_{\hat p_{\text{MLE}}} + \frac{m}{n+m}\cdot p_0 ,
$$

an explicit convex combination: the prior behaves as $m$ extra observations located at $p_0$.

**Rate of prior decay.** Subtracting,

$$
\hat p_{\text{MAP}} - \hat p_{\text{MLE}} = \frac{m}{n+m}\left(p_0 - \hat p_{\text{MLE}}\right) = O\!\left(\frac{1}{n}\right) .
$$

Since the estimator's own standard error is $O(n^{-1/2})$, the prior's contribution is asymptotically negligible compared with sampling noise: it matters at $n \sim m$ and is invisible by $n \sim m^2$. This is the general phenomenon — $\ell$ grows like $n$ while $\ln p(\theta)$ stays $O(1)$.

**Boundary behavior.** For $\alpha = \beta = 1$ (uniform prior) MAP $=$ MLE exactly. For $\alpha,\beta \lt 1$ the prior density diverges at the endpoints and the MAP can be pushed to $0$ or $1$ — a reminder that "uninformative" priors can be strongly informative about the mode.

$$
\boxed{\hat p_{\text{MAP}} = \frac{n\,\hat p_{\text{MLE}} + m\,p_0}{n+m}, \qquad \hat p_{\text{MAP}} - \hat p_{\text{MLE}} = O(1/n)}
$$

*Key takeaway:* A conjugate prior is literally a pile of pseudo-observations, and its weight relative to the data is $m/(n+m)$.

The cell below recomputes every number claimed in Problem L1.6.

In [11]:
alpha, beta_ = 4.0, 2.0
m = alpha + beta_ - 2                             # prior pseudo-count
p0 = (alpha - 1) / m                              # prior mode
print(f"prior pseudo-count m = {m}, prior mode p0 = {p0:.4f}")

print("\n   n    k    MLE      MAP      (n*MLE + m*p0)/(n+m)   MAP - MLE   1/n")
for n in (4, 10, 40, 200, 1000):
    k = int(round(0.25 * n))
    mle = k / n
    mapv = (k + alpha - 1) / (n + alpha + beta_ - 2)
    mix = (n * mle + m * p0) / (n + m)
    print(f"{n:5d}{k:5d}  {mle:.5f}  {mapv:.5f}        {mix:.5f}          "
          f"{mapv-mle:+.5f}   {1/n:.5f}")
    assert np.isclose(mapv, mix)

gaps = np.array([abs((int(round(0.25*n)) + alpha - 1) / (n + m) - 0.25) for n in (100, 1000, 10000)])
print(f"\n|MAP - MLE| at n = 100, 1000, 10000: {gaps}  -> decays like 1/n")
assert gaps[0] / gaps[1] > 5 and gaps[1] / gaps[2] > 5

alpha_u = beta_u = 1.0                            # uniform prior: MAP = MLE exactly
n, k = 17, 5
assert np.isclose((k + alpha_u - 1) / (n + alpha_u + beta_u - 2), k / n)
print(f"uniform prior at n = {n}, k = {k}: MAP = MLE = {k/n:.6f}")

prior pseudo-count m = 4.0, prior mode p0 = 0.7500

   n    k    MLE      MAP      (n*MLE + m*p0)/(n+m)   MAP - MLE   1/n
    4    1  0.25000  0.50000        0.50000          +0.25000   0.25000
   10    2  0.20000  0.35714        0.35714          +0.15714   0.10000
   40   10  0.25000  0.29545        0.29545          +0.04545   0.02500
  200   50  0.25000  0.25980        0.25980          +0.00980   0.00500
 1000  250  0.25000  0.25199        0.25199          +0.00199   0.00100

|MAP - MLE| at n = 100, 1000, 10000: [0.0192 0.002  0.0002]  -> decays like 1/n


uniform prior at n = 17, k = 5: MAP = MLE = 0.294118


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Logistic Regression — Score, Hessian, and Why Separation Diverges

**Statement**

For labels $y_i \in \{0,1\}$ and features $x_i \in\mathbb{R}^d$, the model is $P(y=1\mid x) = \sigma(w^{\top}x)$ with $\sigma(z) = 1/(1+e^{-z})$. Derive the log-likelihood, its gradient and Hessian, prove concavity, and explain what happens under linear separability.

**Intuition**

The Hessian is a negative-weighted Gram matrix, so concavity is automatic; the only way to lose a maximizer is for the weights to vanish, which is exactly what perfect separation does.

**Solution**

**Log-likelihood.** With $\mu_i = \sigma(w^{\top}x_i)$,

$$
\ell(w) = \sum_{i=1}^n \left[y_i\ln\mu_i + (1-y_i)\ln(1-\mu_i)\right] = \sum_{i=1}^n\left[y_i w^{\top}x_i - \ln\left(1 + e^{w^{\top}x_i}\right)\right] ,
$$

the second form obtained by substituting $\ln\frac{\mu}{1-\mu} = w^{\top}x$ and $\ln(1-\mu) = -\ln(1+e^{w^{\top}x})$. Minimizing $-\ell/n$ is exactly binary cross-entropy loss.

**Gradient.** Using $\sigma'(z) = \sigma(z)(1-\sigma(z))$,

$$
\nabla_w \ell = \sum_{i=1}^n \left(y_i - \mu_i\right)x_i = X^{\top}(y - \mu) .
$$

This is the exponential-family moment-matching equation: at the optimum the predicted and observed feature-weighted counts agree.

**Hessian.**

$$
\nabla^2_w \ell = -\sum_{i=1}^n \mu_i(1-\mu_i)\,x_i x_i^{\top} = -X^{\top}WX, \qquad W = \operatorname{diag}\left(\mu_i(1-\mu_i)\right) .
$$

**Concavity.** For any $v \in\mathbb{R}^d$,

$$
v^{\top}\left(\nabla^2\ell\right)v = -\sum_i \mu_i(1-\mu_i)\left(x_i^{\top}v\right)^2 \le 0 ,
$$

since $\mu_i(1-\mu_i) \gt 0$. So $\ell$ is concave and any stationary point is a global maximum; it is *strictly* concave iff $X$ has full column rank and no $\mu_i \in\{0,1\}$.

**Newton step = IRLS.** $w_{t+1} = w_t + (X^{\top}WX)^{-1}X^{\top}(y-\mu)$, which rearranges to a weighted least-squares solve on the working response $z = Xw_t + W^{-1}(y-\mu)$ — the classical IRLS algorithm, and the Fisher-scoring update since the observed and expected Hessians coincide here.

**Separability.** Suppose some $w^\star$ satisfies $y_i = 1 \Rightarrow w^{\star\top}x_i \gt 0$ and $y_i = 0 \Rightarrow w^{\star\top}x_i \lt 0$. Then for $c \gt 0$ the direction $cw^\star$ drives every $\mu_i$ toward the correct $0$ or $1$, so

$$
\ell(cw^\star) \nearrow 0 \quad\text{as } c\to\infty ,
$$

the supremum is never attained, and $\lVert\hat w\rVert\to\infty$. The Hessian degenerates ($W \to 0$), standard errors explode, and training "diverges" with perfect accuracy. Any proper prior fixes it: adding $-\frac{\lambda}{2}\lVert w\rVert^2$ makes the objective strongly concave with a finite maximizer.

$$
\boxed{\nabla\ell = X^{\top}(y-\mu), \quad \nabla^2\ell = -X^{\top}WX \preceq 0, \quad \text{separability} \Rightarrow \lVert\hat w\rVert = \infty}
$$

*Key takeaway:* Logistic regression is concave MLE with a moment-matching optimum; its one pathology, separation, is cured by the tiniest amount of prior.

The cell below recomputes every number claimed in Problem L2.1.

In [12]:
n, d = 200, 3
X = np.column_stack([rng.normal(size=(n, d - 1)), np.ones(n)])
w_star = np.array([1.0, -1.5, 0.4])
y = (rng.random(n) < expit(X @ w_star)).astype(float)

def loglik(w):
    z = X @ w
    return float(np.sum(y * z - np.logaddexp(0.0, z)))

w0 = np.array([0.3, -0.2, 0.1])
mu = expit(X @ w0)
grad = X.T @ (y - mu)
W = mu * (1 - mu)
hess = -(X.T @ (W[:, None] * X))

eps = 1e-6                                        # finite-difference check of both derivatives
gnum = np.array([(loglik(w0 + eps * e) - loglik(w0 - eps * e)) / (2 * eps)
                 for e in np.eye(d)])
hnum = np.zeros((d, d))
for i, e in enumerate(np.eye(d)):
    gp = X.T @ (y - expit(X @ (w0 + eps * e)))
    gm = X.T @ (y - expit(X @ (w0 - eps * e)))
    hnum[:, i] = (gp - gm) / (2 * eps)
print(f"max |analytic grad - finite difference| = {np.abs(grad - gnum).max():.3e}")
print(f"max |analytic Hess - finite difference| = {np.abs(hess - hnum).max():.3e}")
assert np.abs(grad - gnum).max() < 1e-5 and np.abs(hess - hnum).max() < 1e-5

evals = np.linalg.eigvalsh(hess)
print(f"Hessian eigenvalues = {evals}  -> all <= 0, so the log-likelihood is concave")
assert np.all(evals <= 1e-12)

Xs = np.column_stack([rng.normal(size=60), np.ones(60)])
ys = (Xs[:, 0] > 0).astype(float)                 # separable
def fit(Xd, yd, lam, steps):
    w = np.zeros(Xd.shape[1])
    for _ in range(steps):
        w = w + 0.5 * (Xd.T @ (yd - expit(Xd @ w)) / Xd.shape[0] - lam * w)
    return w
norms = [np.linalg.norm(fit(Xs, ys, 0.0, s)) for s in (1000, 10_000, 100_000)]
ridge_norms = [np.linalg.norm(fit(Xs, ys, 0.1, s)) for s in (1000, 10_000, 100_000)]
print(f"\nseparable, no prior : ||w|| = {np.round(norms, 4)}  (still growing)")
print(f"separable, ridge    : ||w|| = {np.round(ridge_norms, 4)}  (converged)")
assert norms[-1] > norms[0] and abs(ridge_norms[-1] - ridge_norms[-2]) < 1e-6

max |analytic grad - finite difference| = 6.762e-09
max |analytic Hess - finite difference| = 6.766e-09
Hessian eigenvalues = [-53.346  -49.6188 -46.1761]  -> all <= 0, so the log-likelihood is concave



separable, no prior : ||w|| = [ 9.9546 24.3894 58.1386]  (still growing)
separable, ridge    : ||w|| = [1.3685 1.3685 1.3685]  (converged)


### Problem L2.2 — OLS Is MLE, Ridge Is MAP

**Statement**

For $y = Xw + \varepsilon$ with $\varepsilon\sim\mathcal{N}(0,\sigma^2 I_n)$, show that MLE gives ordinary least squares and that a Gaussian prior gives ridge regression. Interpret $\lambda$ and analyze the bias-variance trade.

**Intuition**

Gaussian noise makes the log-likelihood a squared norm and a Gaussian prior makes the log-prior another squared norm, so the whole problem is one quadratic. In the SVD basis it decouples into independent scalar shrinkage problems.

**Solution**

**MLE.**

$$
\ell(w) = -\frac{n}{2}\ln(2\pi\sigma^2) - \frac{1}{2\sigma^2}\lVert y - Xw\rVert_2^2 ,
$$

so maximizing $\ell$ minimizes the residual sum of squares — least squares is a *consequence* of assuming Gaussian noise, not an independent principle. Setting $\nabla_w\ell = \frac{1}{\sigma^2}X^{\top}(y-Xw) = 0$:

$$
\hat{w}_{\text{OLS}} = \left(X^{\top}X\right)^{-1}X^{\top}y .
$$

**MAP with $w\sim\mathcal{N}(0,\tau^2 I_d)$.**

$$
\ln p(w\mid y) = -\frac{1}{2\sigma^2}\lVert y-Xw\rVert^2 - \frac{1}{2\tau^2}\lVert w\rVert^2 + \text{const} .
$$

Setting the gradient $\frac{1}{\sigma^2}X^{\top}(y-Xw) - \frac{w}{\tau^2}$ to zero and multiplying by $\sigma^2$:

$$
X^{\top}y = \left(X^{\top}X + \lambda I\right)w, \qquad \lambda = \frac{\sigma^2}{\tau^2} \quad\Longrightarrow\quad \hat w_{\text{ridge}} = \left(X^{\top}X+\lambda I\right)^{-1}X^{\top}y .
$$

**Interpretation of $\lambda$.** It is the *noise-to-prior-variance ratio*. Noisy data (large $\sigma^2$) or a confident prior (small $\tau^2$) both increase shrinkage; $\tau^2\to\infty$ (flat prior) recovers OLS.

**Bias–variance.** In the SVD basis $X = U\Sigma V^{\top}$ with singular values $s_j$, the ridge estimator shrinks each coordinate of the OLS solution by a factor

$$
\frac{s_j^2}{s_j^2+\lambda} \in (0,1) .
$$

Hence

$$
\operatorname{Var}\left(\hat w_{\text{ridge},j}\right) = \frac{\sigma^2 s_j^2}{(s_j^2+\lambda)^2} \lt \frac{\sigma^2}{s_j^2}, \qquad \text{Bias}_j = -\frac{\lambda}{s_j^2+\lambda}w_j .
$$

Directions with small $s_j$ — those the data barely constrain — are shrunk hardest, which is exactly where OLS variance explodes. There always exists $\lambda \gt 0$ with strictly lower total MSE than OLS, the classical justification for shrinkage. Ridge also guarantees invertibility even when $X^{\top}X$ is singular ($d \gt n$), since $X^{\top}X+\lambda I \succeq \lambda I \succ 0$.

$$
\boxed{\hat w_{\text{OLS}} = (X^{\top}X)^{-1}X^{\top}y, \quad \hat w_{\text{ridge}} = (X^{\top}X+\lambda I)^{-1}X^{\top}y, \ \lambda = \sigma^2/\tau^2}
$$

*Key takeaway:* Least squares and weight decay are not modeling choices bolted together — they are the likelihood and the prior of one Gaussian model.

The cell below recomputes every number claimed in Problem L2.2.

In [13]:
n, d, sigma, tau = 120, 5, 1.5, 0.8
X = rng.normal(size=(n, d)) @ np.diag([3.0, 2.0, 1.0, 0.4, 0.15])   # ill-conditioned on purpose
w_true = rng.normal(0, tau, size=d)
y = X @ w_true + rng.normal(0, sigma, size=n)
lam = sigma**2 / tau**2

w_ols = np.linalg.solve(X.T @ X, X.T @ y)
w_ridge = np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)
print(f"lambda = sigma^2/tau^2 = {lam:.6f}")
print(f"OLS   = {w_ols}")
print(f"ridge = {w_ridge}")

U, s, Vt = np.linalg.svd(X, full_matrices=False)   # SVD form: coordinatewise shrinkage
coef_ols = (U.T @ y) / s
coef_ridge = s * (U.T @ y) / (s**2 + lam)
print(f"singular values   = {s}")
print(f"shrinkage factors = {s**2/(s**2+lam)}")
assert np.allclose(Vt.T @ coef_ols, w_ols) and np.allclose(Vt.T @ coef_ridge, w_ridge)

lams = np.linspace(0.0, 20.0, 401)                 # some lambda > 0 strictly beats OLS
bias2 = np.array([np.sum((lam_ / (s**2 + lam_) * (Vt @ w_true)) ** 2) for lam_ in lams])
var = np.array([np.sum(sigma**2 * s**2 / (s**2 + lam_) ** 2) for lam_ in lams])
mse = bias2 + var
best = lams[np.argmin(mse)]
print(f"\nMSE(lambda = 0) = {mse[0]:.5f},  min MSE at lambda = {best:.3f} with {mse.min():.5f}")
assert best > 0 and mse.min() < mse[0]

lambda = sigma^2/tau^2 = 3.515625
OLS   = [-0.9806 -0.18   -0.1767 -0.2136  2.1439]
ridge = [-0.9789 -0.181  -0.1673 -0.2194  0.9375]
singular values   = [31.9453 20.3096 11.461   4.5632  1.6424]
shrinkage factors = [0.9966 0.9915 0.9739 0.8556 0.4342]

MSE(lambda = 0) = 0.96698,  min MSE at lambda = 1.750 with 0.63201


### Problem L2.3 — Cross-Entropy, Softmax Gradients, and Label Smoothing as a Dirichlet Prior

**Statement**

For $K$-class classification with logits $z = f_\theta(x)$ and softmax $p_k = e^{z_k}/\sum_j e^{z_j}$, show that the training loss is negative log-likelihood, derive $\partial\mathcal{L}/\partial z$, and show that label smoothing is MAP under a Dirichlet prior.

**Intuition**

Softmax with cross-entropy is a canonical-link exponential family, so the gradient must be prediction minus target. Label smoothing changes only the target, which is what a Dirichlet prior does to counts.

**Solution**

**Loss is NLL.** The observation model is Categorical$(p)$, so for a one-hot label $y$,

$$
p(y \mid x,\theta) = \prod_{k=1}^K p_k^{y_k}, \qquad -\ln p(y\mid x,\theta) = -\sum_{k=1}^K y_k \ln p_k ,
$$

which is exactly the cross-entropy $H(y, p)$. Averaging over the dataset gives $-\ell(\theta)/n$: minimizing cross-entropy *is* maximum likelihood.

**Gradient with respect to logits.** Write $\mathcal{L} = -\sum_k y_k\left(z_k - \ln\sum_j e^{z_j}\right)$. Then

$$
\frac{\partial\mathcal{L}}{\partial z_m} = -y_m + \left(\sum_k y_k\right)\frac{e^{z_m}}{\sum_j e^{z_j}} = p_m - y_m ,
$$

using $\sum_k y_k = 1$. In vector form,

$$
\nabla_z \mathcal{L} = p - y ,
$$

the same clean "prediction minus target" form as in logistic regression — a general property of exponential families with canonical link, and the reason softmax + cross-entropy trains stably while softmax + squared error does not (the latter's gradient carries a vanishing $p_k(1-p_k)$ factor).

**Label smoothing as Dirichlet MAP.** Consider estimating a categorical distribution $p$ from counts $n_k$. With prior $p \sim \text{Dirichlet}(\alpha,\ldots,\alpha)$,

$$
\ln p(p\mid n) = \sum_k (n_k + \alpha - 1)\ln p_k + \text{const} .
$$

Maximizing subject to $\sum_k p_k = 1$ with a Lagrange multiplier gives the stationarity condition $\frac{n_k+\alpha-1}{p_k} = \nu$, hence

$$
\hat p_k^{\text{MAP}} = \frac{n_k + \alpha - 1}{N + K(\alpha-1)} .
$$

Write $a = \alpha - 1$ and $c = Ka/N$. Dividing through by $N$,

$$
\hat p_k^{\text{MAP}} = \frac{\tfrac{n_k}{N} + \tfrac{c}{K}}{1 + c} = \frac{1}{1+c}\cdot\frac{n_k}{N} + \frac{c}{1+c}\cdot\frac{1}{K} ,
$$

so matching $\frac{1}{1+c} = 1-\epsilon$ forces $c = \frac{\epsilon}{1-\epsilon}$, i.e.

$$
\alpha - 1 = \frac{N\epsilon}{K(1-\epsilon)} ,
$$

and then the second coefficient is $\frac{c}{1+c} = \epsilon$ automatically. With that value — and only that value — the MAP is *exactly* the label-smoothing target

$$
y_k^{\text{LS}} = (1-\epsilon)y_k + \frac{\epsilon}{K} .
$$

The effect is to keep logit gaps finite: without smoothing, the loss is minimized only as $z_{\text{true}} - z_{\text{other}}\to\infty$, driving overconfidence and poor calibration; with smoothing the optimum is $p = y^{\text{LS}}$, so the optimal logit gap is finite:

$$
z_{\text{true}} - z_{\text{other}} = \ln\frac{y^{\text{LS}}_{\text{true}}}{y^{\text{LS}}_{\text{other}}} = \ln\frac{1 - \epsilon + \epsilon/K}{\epsilon/K} = \ln\frac{K(1-\epsilon) + \epsilon}{\epsilon} .
$$

For $K = 5$ and $\epsilon = 0.1$ this is $\ln 45.5 \approx 3.8286$, whereas the unsmoothed optimum is $+\infty$. (The commonly used alternative convention $y_{\text{true}} = 1-\epsilon$, $y_{\text{other}} = \epsilon/(K-1)$ gives $\ln\frac{(1-\epsilon)(K-1)}{\epsilon}$ instead — the gap depends on the convention, so the convention must be stated.)

$$
\boxed{\nabla_z \mathcal{L} = p - y; \quad \text{label smoothing} = \text{Dirichlet}(\alpha)\ \text{MAP},\ \alpha - 1 = \tfrac{N\epsilon}{K(1-\epsilon)}, \quad \text{gap} = \ln\tfrac{K(1-\epsilon)+\epsilon}{\epsilon}}
$$

*Key takeaway:* Classification training is categorical MLE; smoothing tricks are conjugate priors, and calibration problems are missing priors.

The cell below recomputes every number claimed in Problem L2.3.

In [14]:
K, N, eps = 5, 100, 0.1
counts = np.array([50.0, 20.0, 15.0, 10.0, 5.0])

a = N * eps / (K * (1 - eps))                      # alpha - 1 for the exact correspondence
p_map = (counts + a) / (N + K * a)
target = (1 - eps) * counts / N + eps / K
print(f"alpha - 1 = N*eps/(K(1-eps)) = {a:.6f}")
print(f"Dirichlet MAP        = {p_map}")
print(f"label-smoothed target= {target}")
print(f"max |difference|     = {np.abs(p_map - target).max():.3e}")
assert np.abs(p_map - target).max() < 1e-12
print(f"eps recovered from alpha: K(alpha-1)/(N+K(alpha-1)) = {K*a/(N+K*a):.6f}")
assert np.isclose(K * a / (N + K * a), eps)

a_naive = eps * N / K                              # the first-order-only choice
p_naive = (counts + a_naive) / (N + K * a_naive)
print(f"\nwith alpha - 1 = eps*N/K instead: max |difference| = "
      f"{np.abs(p_naive - target).max():.3e}  (O(eps^2), not exact)")
assert 1e-6 < np.abs(p_naive - target).max() < 1e-2

def ls_loss(z):                                    # optimal logit gap under smoothing
    yls = np.full(K, eps / K)
    yls[0] += 1 - eps
    logp = z - logsumexp(z)
    return -float(yls @ logp)

res = optimize.minimize(lambda t: ls_loss(np.concatenate([[t[0]], np.zeros(K - 1)])),
                        np.array([1.0]), method="Nelder-Mead", options={"xatol": 1e-10})
gap_pred = np.log((K * (1 - eps) + eps) / eps)
print(f"\noptimal logit gap: numerical {res.x[0]:.8f}, formula ln((K(1-eps)+eps)/eps) = {gap_pred:.8f}")
assert np.isclose(res.x[0], gap_pred, atol=1e-5)

z = rng.normal(size=K)                             # gradient of cross-entropy wrt logits is p - y
p = np.exp(z - logsumexp(z))
yh = np.zeros(K); yh[2] = 1.0
num = np.array([( -(yh @ (z + 1e-6*e - logsumexp(z + 1e-6*e)))
                  + (yh @ (z - 1e-6*e - logsumexp(z - 1e-6*e))) ) / (2e-6)
                for e in np.eye(K)])
print(f"max |analytic (p - y) - finite difference| = {np.abs((p - yh) - num).max():.3e}")
assert np.abs((p - yh) - num).max() < 1e-6

alpha - 1 = N*eps/(K(1-eps)) = 2.222222
Dirichlet MAP        = [0.47  0.2   0.155 0.11  0.065]
label-smoothed target= [0.47  0.2   0.155 0.11  0.065]
max |difference|     = 5.551e-17
eps recovered from alpha: K(alpha-1)/(N+K(alpha-1)) = 0.100000

with alpha - 1 = eps*N/K instead: max |difference| = 2.727e-03  (O(eps^2), not exact)

optimal logit gap: numerical 3.82864137, formula ln((K(1-eps)+eps)/eps) = 3.82864140
max |analytic (p - y) - finite difference| = 1.342e-10


### Problem L2.4 — Lasso from a Laplace Prior and the Soft-Thresholding Solution

**Statement**

Derive the lasso objective as MAP under a Laplace prior and solve it in closed form for an orthonormal design $X^{\top}X = I$. Explain why lasso is sparse and ridge is not.

**Intuition**

The Laplace prior contributes a penalty whose subgradient at the origin is a whole interval, so a range of data values maps to exactly zero. Ridge's penalty has a single subgradient there, namely $0$, so it never selects.

**Solution**

**MAP objective.** With $y = Xw+\varepsilon$, $\varepsilon\sim\mathcal{N}(0,\sigma^2I)$, and independent priors $p(w_j) = \frac{1}{2b}e^{-\vert w_j\vert/b}$,

$$
-\ln p(w\mid y) = \frac{1}{2\sigma^2}\lVert y-Xw\rVert_2^2 + \frac{1}{b}\sum_{j}\vert w_j\vert + \text{const} ,
$$

so multiplying by $2\sigma^2$,

$$
\hat w_{\text{lasso}} = \arg\min_w \left\{\lVert y - Xw\rVert_2^2 + \lambda\lVert w\rVert_1\right\}, \qquad \lambda = \frac{2\sigma^2}{b} .
$$

**Orthonormal design.** With $X^{\top}X = I$ and $\tilde w = X^{\top}y$ (the OLS solution), expand:

$$
\lVert y-Xw\rVert^2 = \lVert y\rVert^2 - 2w^{\top}\tilde w + \lVert w\rVert^2 ,
$$

so the objective separates coordinatewise into

$$
\min_{w_j}\ \left\{w_j^2 - 2w_j\tilde w_j + \lambda\vert w_j\vert\right\} .
$$

**Subgradient condition.** The objective is convex but non-differentiable at $0$; stationarity requires $0 \in 2w_j - 2\tilde w_j + \lambda\,\partial\vert w_j\vert$ where $\partial\vert w_j\vert = \{\operatorname{sign}(w_j)\}$ for $w_j\ne0$ and $[-1,1]$ at $0$.

- If $w_j \gt 0$: $w_j = \tilde w_j - \lambda/2$, consistent when $\tilde w_j \gt \lambda/2$.
- If $w_j \lt 0$: $w_j = \tilde w_j + \lambda/2$, consistent when $\tilde w_j \lt -\lambda/2$.
- If $\vert \tilde w_j\vert \le \lambda/2$: $0$ is optimal, since $2\tilde w_j \in \lambda[-1,1]$.

Combining gives **soft thresholding**:

$$
\hat w_j = \operatorname{sign}(\tilde w_j)\max\left(\vert \tilde w_j\vert - \frac{\lambda}{2},\ 0\right) .
$$

**Why sparsity.** The whole interval $[-1,1]$ of subgradients at $0$ means a *range* of data values maps to exactly $w_j = 0$ — the Laplace prior's kink creates a set of positive measure on which the optimum is exactly zero. Ridge, by contrast, has the smooth penalty $\lambda w_j^2$ with derivative $2\lambda w_j \to 0$ at the origin, so its solution $\hat w_j = \tilde w_j/(1+\lambda)$ shrinks proportionally but never reaches zero.

**Caution on the Bayesian reading.** The lasso estimate is the posterior *mode* of the Laplace-prior model; the posterior *mean* is never sparse (the posterior has a density, so $P(w_j = 0) = 0$). Sparsity is an artifact of mode-seeking, which is one more reason to view MAP as regularized optimization rather than as Bayesian inference.

$$
\boxed{\hat w_j = \operatorname{sign}(\tilde w_j)\left(\vert \tilde w_j\vert - \tfrac{\lambda}{2}\right)_+ \ \text{(lasso)} \quad\text{vs}\quad \hat w_j = \tfrac{\tilde w_j}{1+\lambda}\ \text{(ridge)}}
$$

*Key takeaway:* Sparsity comes from a non-differentiable prior at the origin; smooth priors shrink but never select.

The cell below recomputes every number claimed in Problem L2.4.

In [15]:
d, lam = 6, 1.4
Q, _ = np.linalg.qr(rng.normal(size=(d, d)))       # orthonormal design: X^T X = I
X = Q
w_true = np.array([2.0, -0.5, 0.3, 0.0, 1.1, -0.05])
y = X @ w_true + rng.normal(0, 0.2, size=d)
w_tilde = X.T @ y

soft = np.sign(w_tilde) * np.maximum(np.abs(w_tilde) - lam / 2, 0.0)
obj = lambda w: np.sum((y - X @ w) ** 2) + lam * np.sum(np.abs(w))
num = optimize.minimize(obj, np.zeros(d), method="Powell",
                        options={"xtol": 1e-12, "ftol": 1e-12, "maxiter": 200000})
print(f"OLS coefficients   = {w_tilde}")
print(f"soft threshold     = {soft}")
print(f"numerical minimizer= {num.x}")
print(f"max |difference|   = {np.abs(soft - num.x).max():.3e}")
print(f"objective: closed form {obj(soft):.10f}, numerical {num.fun:.10f}")
assert np.abs(soft - num.x).max() < 1e-4 and obj(soft) <= num.fun + 1e-9
print(f"\nexact zeros: lasso {int(np.sum(soft == 0))} of {d}, "
      f"ridge {int(np.sum(w_tilde / (1 + lam) == 0))} of {d}")
assert np.sum(soft == 0) > 0 and np.sum(w_tilde / (1 + lam) == 0) == 0

OLS coefficients   = [ 1.9983 -0.5347  0.3595 -0.0403  1.3161  0.0649]
soft threshold     = [ 1.2983 -0.      0.     -0.      0.6161  0.    ]
numerical minimizer= [1.2983 0.     0.     0.     0.6161 0.    ]
max |difference|   = 6.676e-09
objective: closed form 4.0811809498, numerical 4.0811809498

exact zeros: lasso 4 of 6, ridge 0 of 6


### Problem L2.5 — Estimating a Decay Constant from Timestamped Events

**Statement**

A detector records $n$ decay times $t_1,\ldots,t_n$ from an $\text{Exponential}(\lambda)$ source, but only events with $t \lt T$ are recorded (truncation at the end of the run). Derive the MLE of $\lambda$ accounting for truncation, and compare with the naive estimator.

**Intuition**

The detector never reports the events it missed, so the likelihood must be the *conditional* density given $t \lt T$. Forgetting the normalizer means fitting a distribution that has been silently truncated, which inflates the estimated rate.

**Solution**

**Truncated density.** Conditioning on $T_i \lt T$ renormalizes the exponential:

$$
f(t\mid\lambda, t \lt T) = \frac{\lambda e^{-\lambda t}}{1 - e^{-\lambda T}}, \qquad 0 \lt t \lt T .
$$

**Log-likelihood.**

$$
\ell(\lambda) = n\ln\lambda - \lambda\sum_{i=1}^n t_i - n\ln\left(1 - e^{-\lambda T}\right) .
$$

**Score equation.** Differentiating,

$$
\ell'(\lambda) = \frac{n}{\lambda} - \sum_i t_i - n\,\frac{T e^{-\lambda T}}{1-e^{-\lambda T}} = 0 ,
$$

which rearranges to the moment-matching condition

$$
\bar{t} = \frac{1}{\lambda} - \frac{Te^{-\lambda T}}{1-e^{-\lambda T}} = E\left[T_i \mid T_i \lt T\right] .
$$

The right side is the mean of the truncated exponential, so the MLE equates the observed and model-implied conditional means. No closed form exists; solve by Newton's method with the untruncated $\hat\lambda_0 = 1/\bar t$ as the starting point.

**Bias of the naive estimator.** Ignoring truncation and using $\hat\lambda_{\text{naive}} = 1/\bar{t}$ *overestimates* $\lambda$, because truncation removes the long times and shrinks $\bar t$ below $1/\lambda$. Quantitatively, with $u = \lambda T$,

$$
E\left[t \mid t \lt T\right] = \frac{1}{\lambda}\left(1 - \frac{ue^{-u}}{1-e^{-u}}\right) ,
$$

so for $u = 1$ (run length equal to one lifetime) the factor is $1 - \frac{0.3679}{0.6321} = 0.418$: the naive estimate is inflated by $1/0.4180 \approx 2.392\times$. For $u = 5$ the factor is $0.966$ and the bias is under $4\%$ — run long relative to the lifetime and truncation stops mattering.

**Uncertainty.** The Fisher information for the truncated model is $I(\lambda) = \frac{1}{\lambda^2} - \frac{T^2 e^{-\lambda T}}{(1-e^{-\lambda T})^2}$, strictly smaller than the untruncated $1/\lambda^2$: censoring destroys information, so standard errors widen.

$$
\boxed{\bar{t} = \frac{1}{\hat\lambda} - \frac{Te^{-\hat\lambda T}}{1-e^{-\hat\lambda T}}; \ \text{ignoring truncation inflates } \hat\lambda}
$$

*Key takeaway:* The likelihood must describe how data were *observed*, not merely how they were generated — censoring and selection belong inside the model.

The cell below recomputes every number claimed in Problem L2.5.

In [16]:
from scipy.optimize import brentq

lam_true, T, n = 1.0, 1.0, 20_000                  # run length equal to one lifetime: u = 1
t = rng.exponential(1 / lam_true, size=4 * n)
t = t[t < T][:n]                                   # keep only the events the detector sees

u = lam_true * T
factor = 1 - u * np.exp(-u) / (1 - np.exp(-u))
print(f"u = lambda*T = {u}:  E[t | t < T] = {factor:.6f}/lambda   (hand 0.418)")
print(f"observed mean of truncated sample = {t.mean():.6f}, predicted {factor/lam_true:.6f}")
assert np.isclose(t.mean(), factor / lam_true, rtol=0.02)

score = lambda L: n / L - t.sum() - n * T * np.exp(-L * T) / (1 - np.exp(-L * T))
lam_trunc = brentq(score, 1e-6, 100.0)
lam_naive = 1 / t.mean()
print(f"\ntruncation-aware MLE = {lam_trunc:.5f}   (true {lam_true})")
print(f"naive 1/tbar         = {lam_naive:.5f}   inflation {lam_naive/lam_true:.4f}x "
      f"(hand ~{1/factor:.2f}x)")
assert abs(lam_trunc - lam_true) < 0.05 and np.isclose(lam_naive, 1 / factor, rtol=0.03)

I_trunc = 1 / lam_true**2 - T**2 * np.exp(-lam_true * T) / (1 - np.exp(-lam_true * T)) ** 2
h = 1e-5                                           # numerical -d2/dlam2 of the per-event loglik
f = lambda L: np.log(L) - L * t - np.log1p(-np.exp(-L * T))
I_num = -((f(lam_true + h) - 2 * f(lam_true) + f(lam_true - h)) / h**2).mean()
print(f"\nI_truncated = {I_trunc:.6f} (formula), {I_num:.6f} (numerical), "
      f"vs untruncated 1/lambda^2 = {1/lam_true**2:.6f}")
assert np.isclose(I_trunc, I_num, rtol=1e-3) and I_trunc < 1 / lam_true**2

for uu in (1.0, 5.0):
    fac = 1 - uu * np.exp(-uu) / (1 - np.exp(-uu))
    print(f"u = {uu}: mean factor {fac:.4f}, naive inflation {1/fac:.4f}x")

u = lambda*T = 1.0:  E[t | t < T] = 0.418023/lambda   (hand 0.418)
observed mean of truncated sample = 0.416163, predicted 0.418023

truncation-aware MLE = 1.02348   (true 1.0)
naive 1/tbar         = 2.40291   inflation 2.4029x (hand ~2.39x)

I_truncated = 0.079326 (formula), 0.079327 (numerical), vs untruncated 1/lambda^2 = 1.000000
u = 1.0: mean factor 0.4180, naive inflation 2.3922x
u = 5.0: mean factor 0.9661, naive inflation 1.0351x


### Problem L2.6 — EM for a Two-Component Mixture — Separating a Spectral Line from Background

**Statement**

A spectrometer records photon energies from a narrow emission line sitting on a broad background, so the energy of each photon is drawn from $p(x) = \pi\,\mathcal{N}(x\mid\mu_1,\sigma_1^2) + (1-\pi)\mathcal{N}(x\mid\mu_2,\sigma_2^2)$ with the line's provenance unrecorded. Derive the E- and M-steps that estimate the line centre $\mu_1$, its width $\sigma_1$ and the line fraction $\pi$, and explain why the likelihood is unbounded.

**Intuition**

The complete-data problem — knowing which line each photon came from — has a one-line solution, so replace the unknown labels by their posterior probabilities and solve the weighted problem instead. The catch is that a component can always shrink onto a single photon.

**Solution**

**Complete-data model.** Introduce latent labels $z_i \in\{1,2\}$ with $P(z_i = 1) = \pi$. The complete-data log-likelihood is

$$
\ell_c = \sum_{i=1}^n \sum_{k=1}^2 \mathbf{1}\{z_i = k\}\left[\ln \pi_k + \ln\mathcal{N}\left(x_i \mid \mu_k, \sigma_k^2\right)\right], \qquad \pi_1 = \pi,\ \pi_2 = 1-\pi .
$$

**E-step (responsibilities).** By Bayes' rule the posterior over the latent label is

$$
\gamma_{ik} = P\left(z_i = k \mid x_i, \theta^{(t)}\right) = \frac{\pi_k\,\mathcal{N}\left(x_i\mid\mu_k,\sigma_k^2\right)}{\sum_{j=1}^2 \pi_j\,\mathcal{N}\left(x_i\mid\mu_j,\sigma_j^2\right)} ,
$$

computed in log space with `logsumexp` for stability. Substituting into $\ell_c$ replaces the indicators by $\gamma_{ik}$.

**M-step.** Maximize $Q(\theta) = \sum_{i,k}\gamma_{ik}\left[\ln\pi_k + \ln\mathcal{N}(x_i\mid\mu_k,\sigma_k^2)\right]$. Writing $N_k = \sum_i \gamma_{ik}$ and differentiating each parameter separately (the $\gamma$ are held fixed):

$$
\mu_k^{(t+1)} = \frac{1}{N_k}\sum_{i=1}^n \gamma_{ik}x_i, \qquad \left(\sigma_k^2\right)^{(t+1)} = \frac{1}{N_k}\sum_{i=1}^n \gamma_{ik}\left(x_i - \mu_k^{(t+1)}\right)^2, \qquad \pi^{(t+1)} = \frac{N_1}{n} ,
$$

the last obtained with a Lagrange multiplier for $\sum_k\pi_k = 1$. These are the ordinary MLE formulas with each observation counted fractionally according to its responsibility — soft $k$-means, which becomes hard $k$-means in the limit $\sigma^2 \to 0$ with equal, fixed variances.

**Monotonicity.** By Proof 5.8, $\ell(\theta^{(t+1)}) \ge \ell(\theta^{(t)})$ at every iteration; the sequence converges to a stationary point but not necessarily the global optimum.

**Unbounded likelihood.** Set $\mu_1 = x_1$ exactly and let $\sigma_1 \to 0$. The first component's density at $x_1$ is $\frac{1}{\sqrt{2\pi}\sigma_1}\to\infty$, while the remaining points are still explained by component 2 with bounded density. Hence

$$
\ell(\theta) \ \ge\ \ln\left(\frac{\pi}{\sqrt{2\pi}\sigma_1}\right) + \text{bounded} \ \longrightarrow\ +\infty .
$$

The global maximum of the likelihood is $+\infty$ at a degenerate spike, so "the MLE" does not exist for mixtures. Practical fixes — variance floors, or an inverse-gamma prior making the objective a MAP — are the standard remedy, and multiple random restarts handle the remaining local optima.

$$
\boxed{\gamma_{ik}\ \text{(E-step)}; \ \mu_k = \tfrac{\sum_i\gamma_{ik}x_i}{N_k},\ \sigma_k^2 = \tfrac{\sum_i\gamma_{ik}(x_i-\mu_k)^2}{N_k},\ \pi = \tfrac{N_1}{n}\ \text{(M-step)}}
$$

*Key takeaway:* EM converts an intractable marginal likelihood into weighted complete-data MLEs; for mixtures the likelihood is unbounded, so a prior is not optional.

The cell below recomputes every number claimed in Problem L2.6.

In [17]:
# A spectrometer records photon energies: a narrow line on top of a broad background.
n_ph = 1500
is_line = rng.random(n_ph) < 0.30
energy = np.where(is_line, rng.normal(511.0, 1.2, n_ph), rng.normal(505.0, 6.0, n_ph))

def em_two(x, mu, sd, pi, iters=120, floor=1e-3):
    hist = []
    for _ in range(iters):
        a = np.log(pi) + stats.norm.logpdf(x, mu[0], sd[0])
        b = np.log1p(-pi) + stats.norm.logpdf(x, mu[1], sd[1])
        m = logsumexp(np.stack([a, b]), axis=0)
        hist.append(m.sum())
        g = np.exp(a - m)
        N1 = g.sum()
        mu = np.array([(g * x).sum() / N1, ((1 - g) * x).sum() / (x.size - N1)])
        sd = np.array([max(np.sqrt((g * (x - mu[0])**2).sum() / N1), floor),
                       max(np.sqrt(((1 - g) * (x - mu[1])**2).sum() / (x.size - N1)), floor)])
        pi = N1 / x.size
    return np.array(hist), mu, sd, pi

hist, mu, sd, pi = em_two(energy, np.array([508.0, 512.0]), np.array([5.0, 5.0]), 0.5)

# A mixture is identified only up to a permutation of its components: nothing in the
# likelihood distinguishes "component 0" from "component 1", so EM may return them in
# either order. Match them by a property instead — here the spectral line is the narrow one.
line, backg = (0, 1) if sd[0] < sd[1] else (1, 0)
w_line = pi if line == 0 else 1 - pi

print(f"line   : centre {mu[line]:8.3f} keV, width {sd[line]:6.3f}   (true 511.0, 1.2)")
print(f"backgr.: centre {mu[backg]:8.3f} keV, width {sd[backg]:6.3f}   (true 505.0, 6.0)")
print(f"line fraction = {w_line:.4f}   (true 0.30)")
print(f"loglik {hist[0]:.4f} -> {hist[-1]:.4f}; monotone: {bool(np.all(np.diff(hist) >= -1e-9))}")

assert np.all(np.diff(hist) >= -1e-9)          # EM never decreases the log-likelihood
assert abs(mu[line] - 511.0) < 0.6 and abs(w_line - 0.30) < 0.08

for s in (1e-2, 1e-4, 1e-6, 1e-8):                 # unbounded likelihood: spike on one photon
    a = np.log(0.5) + stats.norm.logpdf(energy, energy[0], s)
    b = np.log(0.5) + stats.norm.logpdf(energy, energy.mean(), energy.std())
    print(f"sigma_1 = {s:8.1e}: loglik = {logsumexp(np.stack([a, b]), axis=0).sum():12.4f}")

line   : centre  510.783 keV, width  1.246   (true 511.0, 1.2)
backgr.: centre  504.850 keV, width  5.879   (true 505.0, 6.0)
line fraction = 0.3160   (true 0.30)
loglik -5008.1083 -> -4540.4130; monotone: True
sigma_1 =  1.0e-02: loglik =   -5736.1256
sigma_1 =  1.0e-04: loglik =   -5750.4612
sigma_1 =  1.0e-06: loglik =   -5745.8560
sigma_1 =  1.0e-08: loglik =   -5741.2509


## L3 — Challenge Proofs

### Problem L3.1 — Efficiency, and the Bound for a Nonlinear Function

**Statement**

(a) Show that $\bar{X}$ attains the Cramér–Rao bound for the Gaussian mean. (b) Derive the bound for estimating $g(\lambda) = e^{-\lambda}$ from a Poisson sample and compare with the MLE's asymptotic variance. (c) Explain how ridge regression can have lower MSE than any unbiased estimator without contradiction.

**Intuition**

The Cramér–Rao bound constrains a class, and every practical shrinkage estimator escapes by leaving the class. Where the target is a nonlinear function of the parameter, even the class's best member misses the bound at finite $n$.

**Solution**

**(a) Gaussian mean.** For $\mathcal{N}(\mu,\sigma^2)$ with known $\sigma^2$, $\ln p = -\frac{(x-\mu)^2}{2\sigma^2}+\text{const}$, so $s = \frac{x-\mu}{\sigma^2}$ and

$$
I(\mu) = \operatorname{Var}\left(\frac{X-\mu}{\sigma^2}\right) = \frac{1}{\sigma^2} .
$$

The bound is $\frac{1}{nI} = \frac{\sigma^2}{n}$, and $\operatorname{Var}(\bar X) = \sigma^2/n$ exactly. Equality in Cauchy–Schwarz is confirmed by $s_n = \frac{n}{\sigma^2}(\bar X - \mu)$, affine in the estimator.

**(b) Poisson, $g(\lambda) = e^{-\lambda}$.** From Problem L1.3, $I(\lambda) = 1/\lambda$. With $g'(\lambda) = -e^{-\lambda}$, the bound for unbiased estimators of $g(\lambda)$ is

$$
\operatorname{Var}(T)\ \ge\ \frac{\left[g'(\lambda)\right]^2}{nI(\lambda)} = \frac{\lambda e^{-2\lambda}}{n} .
$$

The MLE is $\widehat{g} = e^{-\bar X}$ (invariance), and the delta method gives asymptotic variance $\left[g'(\lambda)\right]^2\frac{\lambda}{n} = \frac{\lambda e^{-2\lambda}}{n}$ — attaining the bound asymptotically. At finite $n$ it does not, because $e^{-\bar X}$ is biased (Jensen: $E[e^{-\bar X}] \gt e^{-\lambda}$ by convexity). The unique unbiased estimator, $T = \left(1 - \frac1n\right)^{\sum_i X_i}$, has strictly larger variance than $\lambda e^{-2\lambda}/n$ at finite $n$: no unbiased estimator is efficient here, only asymptotically so.

**(c) Ridge versus the bound.** The Cramér–Rao bound constrains only estimators satisfying $E_\theta[T] = \theta$ for *every* $\theta$. Ridge is biased, so it is outside the constrained class. Decompose

$$
\text{MSE} = \text{Bias}^2 + \text{Variance} .
$$

In the SVD basis (Problem L2.2) ridge has bias $-\frac{\lambda}{s_j^2+\lambda}w_j$ (quadratic in $\lambda$ near $0$) and variance $\frac{\sigma^2 s_j^2}{(s_j^2+\lambda)^2}$. Differentiating the total MSE at $\lambda = 0$ gives

$$
\frac{d}{d\lambda}\text{MSE}\Big|_{\lambda=0} = -\frac{2\sigma^2}{s_j^4} \lt 0 ,
$$

so a small positive $\lambda$ *strictly decreases* MSE: the first-order variance reduction beats the second-order bias increase. This is the same mechanism as the James–Stein estimator dominating $\bar{X}$ for a Gaussian mean in dimension $d \ge 3$.

$$
\boxed{\text{CRLB binds unbiased estimators only; bias buys variance at first order}}
$$

*Key takeaway:* Efficiency bounds are conditional on unbiasedness — a constraint no practical high-dimensional method accepts.

The cell below recomputes every number claimed in Problem L3.1.

In [18]:
mu0, sigma, n, reps = 1.0, 2.0, 30, 300_000       # (a) Gaussian mean attains the bound
xb = rng.normal(mu0, sigma, size=(reps, n)).mean(axis=1)
print(f"Var(Xbar) = {xb.var(ddof=1):.6f}   CRLB = sigma^2/n = {sigma**2/n:.6f}")
assert np.isclose(xb.var(ddof=1), sigma**2 / n, rtol=0.02)

lam0, n2 = 2.0, 4                                  # (b) g(lambda) = e^-lambda
crlb_g = lam0 * np.exp(-2 * lam0) / n2
t = 1 - 1 / n2
var_T = np.exp(n2 * lam0 * (t**2 - 1)) - np.exp(2 * n2 * lam0 * (t - 1))
print(f"\nCRLB for e^-lambda        = {crlb_g:.8f}")
print(f"Var of unique unbiased T  = {var_T:.8f}   ratio {var_T/crlb_g:.4f} > 1")
S = rng.poisson(n2 * lam0, size=400_000)
print(f"E[T] (simulated)          = {np.mean(t ** S):.6f}   target e^-lambda = {np.exp(-lam0):.6f}")
assert var_T > crlb_g and np.isclose(np.mean(t ** S), np.exp(-lam0), rtol=0.02)

for nn in (20, 100, 1000):                         # the MLE attains the bound asymptotically
    lam_hat = rng.poisson(lam0, size=(200_000, nn)).mean(axis=1)
    g_hat = np.exp(-lam_hat)
    print(f"n = {nn:5d}: n*Var(e^-lambda-hat) = {nn*g_hat.var():.6f}   "
          f"predicted lambda e^-2lambda = {lam0*np.exp(-2*lam0):.6f}")

s2, w = 4.0, 1.0                                   # (c) d/dlambda MSE at 0 is negative
mse = lambda L: (L / (s2 + L)) ** 2 * w**2 + sigma**2 * s2 / (s2 + L) ** 2
h = 1e-6
print(f"\nd/dlambda MSE at lambda = 0: numerical {(mse(h)-mse(0))/h:+.6f}, "
      f"formula -2 sigma^2/s^4 = {-2*sigma**2/s2**2:+.6f}")
assert (mse(h) - mse(0)) / h < 0

Var(Xbar) = 0.133142   CRLB = sigma^2/n = 0.133333

CRLB for e^-lambda        = 0.00915782
Var of unique unbiased T  = 0.01188174   ratio 1.2974 > 1
E[T] (simulated)          = 0.134916   target e^-lambda = 0.135335


n =    20: n*Var(e^-lambda-hat) = 0.040306   predicted lambda e^-2lambda = 0.036631


n =   100: n*Var(e^-lambda-hat) = 0.037310   predicted lambda e^-2lambda = 0.036631


n =  1000: n*Var(e^-lambda-hat) = 0.036724   predicted lambda e^-2lambda = 0.036631

d/dlambda MSE at lambda = 0: numerical -0.500000, formula -2 sigma^2/s^4 = -0.500000


### Problem L3.2 — Misspecification and the Sandwich Covariance

**Statement**

Suppose the data are generated by $p_0$ but you fit a model family $\{p_\theta\}$ that does not contain $p_0$. Determine what the MLE converges to and derive the correct asymptotic covariance.

**Intuition**

The information equality was derived from $\int p_\theta = 1$, so it is a statement about the *model*, not the data. When the data come from elsewhere the two sides of the equality drift apart and both must be estimated.

**Solution**

**The limit.** Repeating Proof 5.5 without assuming $p_0 = p_{\theta_0}$: by the LLN,

$$
\frac{1}{n}\ell_n(\theta) \xrightarrow{a.s.} E_{p_0}\left[\ln p_\theta(X)\right] = -H(p_0) - D_{\mathrm{KL}}\left(p_0 \parallel p_\theta\right) .
$$

Since $H(p_0)$ does not depend on $\theta$, the maximizer converges to the **pseudo-true parameter**

$$
\theta^\star = \arg\min_\theta D_{\mathrm{KL}}\left(p_0 \parallel p_\theta\right) ,
$$

the KL-closest member of the family. The MLE remains well-behaved; it simply targets a projection rather than a truth. This is exactly what a neural network trained by cross-entropy does — the model class never contains the data-generating process.

**Covariance.** Re-run Proof 5.6 with expectations under $p_0$ and define

$$
H = -E_{p_0}\left[\nabla^2 \ln p_{\theta^\star}(X)\right] \ \ (\text{expected negative Hessian}), \qquad J = E_{p_0}\left[s\,s^{\top}\right] \ \ (\text{score covariance at } \theta^\star) .
$$

Note $E_{p_0}[s(\theta^\star;X)] = 0$ still holds, because $\theta^\star$ is the stationary point of the population objective — but only at $\theta^\star$, and the mean-zero property no longer follows from the normalization identity. The expansion gives

$$
\sqrt{n}\left(\hat\theta_n - \theta^\star\right) = H^{-1}\cdot\frac{1}{\sqrt n}\sum_i s(\theta^\star;X_i) + o_P(1) \ \xrightarrow{d}\ \mathcal{N}\left(0,\ H^{-1}JH^{-1}\right) .
$$

**The sandwich.** When the model is correct, the information equality forces $H = J = I(\theta_0)$ and the sandwich collapses to $I^{-1}$. Under misspecification $H \ne J$ and using the naive $H^{-1}/n$ misstates the standard errors — typically *understating* them, producing confidence intervals with far below nominal coverage.

**Practice.** Estimate both pieces empirically:

$$
\hat H = -\frac1n\sum_i \nabla^2\ln p_{\hat\theta}(x_i), \qquad \hat J = \frac1n\sum_i s(\hat\theta;x_i)s(\hat\theta;x_i)^{\top} ,
$$

giving the Huber–White robust covariance $\hat H^{-1}\hat J\hat H^{-1}/n$. Comparing $\hat H$ with $\hat J$ is itself a specification test (White's test); a large discrepancy is direct evidence that the model is wrong.

$$
\boxed{\hat\theta_n \to \theta^\star = \arg\min_\theta D_{\mathrm{KL}}(p_0 \parallel p_\theta), \quad \operatorname{ACov} = H^{-1}JH^{-1}}
$$

*Key takeaway:* Under misspecification the MLE still converges — to the KL projection — but the information equality fails and honest errors require the sandwich.

The cell below recomputes every number claimed in Problem L3.2.

In [19]:
# True law: Student-t errors. Fitted family: Gaussian with unknown mean, unit variance.
n, reps, df = 200, 4000, 3.0
theta_star = 0.0                                   # KL-closest Gaussian mean is the true mean, 0

H = 1.0                                            # -E[d2 log p] for the unit-variance Gaussian
J = stats.t.var(df)                                # Var(score) = Var(X) under the true law
print(f"H = {H:.6f}, J = Var_t(X) = {J:.6f}  ->  information equality fails, H != J")
print(f"sandwich variance H^-1 J H^-1 = {J / H**2:.6f}, naive H^-1 = {1/H:.6f}")

theta_hat = stats.t.rvs(df, size=(reps, n), random_state=int(rng.integers(1 << 31))).mean(axis=1)
print(f"\nn*Var(theta-hat) = {n*theta_hat.var(ddof=1):.6f}   sandwich prediction {J/H**2:.6f}"
      f"   naive prediction {1/H:.6f}")
assert np.isclose(n * theta_hat.var(ddof=1), J / H**2, rtol=0.08)

z = stats.norm.ppf(0.975)
se_naive = np.sqrt(1 / (n * H))
se_sand = np.sqrt(J / H**2 / n)
cov_naive = np.mean(np.abs(theta_hat - theta_star) < z * se_naive)
cov_sand = np.mean(np.abs(theta_hat - theta_star) < z * se_sand)
print(f"coverage of a nominal 95% interval: naive {cov_naive:.4f}, sandwich {cov_sand:.4f}")
assert cov_naive < 0.94 and cov_sand > 0.93

H = 1.000000, J = Var_t(X) = 3.000000  ->  information equality fails, H != J
sandwich variance H^-1 J H^-1 = 3.000000, naive H^-1 = 1.000000

n*Var(theta-hat) = 2.915308   sandwich prediction 3.000000   naive prediction 1.000000
coverage of a nominal 95% interval: naive 0.7530, sandwich 0.9517


### Problem L3.3 — MAP Is Not Reparameterization-Invariant

**Statement**

Let $\theta$ have posterior density $p(\theta\mid x)$ and let $\phi = g(\theta)$ be a smooth monotone reparameterization. Show that $\hat\phi_{\text{MAP}} \ne g\left(\hat\theta_{\text{MAP}}\right)$ in general, give an explicit example, and identify which estimators do respect reparameterization.

**Intuition**

A density is a measure divided by a reference measure, and changing coordinates changes the reference measure by a Jacobian. Anything defined by maximizing a density therefore moves; anything defined by integrating one does not.

**Solution**

**Change of variables.** With $\theta = g^{-1}(\phi)$, the posterior density of $\phi$ carries a Jacobian:

$$
p_\phi(\phi\mid x) = p_\theta\left(g^{-1}(\phi)\mid x\right)\left\vert \frac{d g^{-1}(\phi)}{d\phi}\right\vert .
$$

Taking logs and differentiating,

$$
\frac{d}{d\phi}\ln p_\phi = \underbrace{\frac{d}{d\phi}\ln p_\theta\left(g^{-1}(\phi)\mid x\right)}_{\text{zero at } \phi = g(\hat\theta_{\text{MAP}})} + \frac{d}{d\phi}\ln\left\vert \frac{dg^{-1}}{d\phi}\right\vert .
$$

At $\phi = g(\hat\theta_{\text{MAP}})$ the first term vanishes but the second generally does not, so that point is *not* stationary for $p_\phi$. The mode moves; only affine $g$ (constant Jacobian) leaves it fixed.

**Explicit example.** Let $\theta \mid x \sim \text{Beta}(3,2)$, with density $\propto \theta^2(1-\theta)$ on $(0,1)$. Its mode is

$$
\hat\theta_{\text{MAP}} = \frac{a-1}{a+b-2} = \frac{2}{3} \approx 0.667 .
$$

Reparameterize by the log-odds $\phi = \ln\frac{\theta}{1-\theta}$, so $\theta = \sigma(\phi)$ and $\frac{d\theta}{d\phi} = \theta(1-\theta)$. The transformed density is

$$
p_\phi(\phi) \propto \theta^2(1-\theta)\cdot\theta(1-\theta) = \theta^3(1-\theta)^2, \qquad \theta = \sigma(\phi) .
$$

Maximizing $3\ln\theta + 2\ln(1-\theta)$ over $\theta$ gives $\frac{3}{\theta} = \frac{2}{1-\theta}$, i.e. $\theta = 3/5 = 0.6$. Hence

$$
\hat\phi_{\text{MAP}} = \ln\frac{0.6}{0.4} = 0.405 \qquad \text{whereas} \qquad g\left(\hat\theta_{\text{MAP}}\right) = \ln\frac{2/3}{1/3} = 0.693 .
$$

Mapping back, the logit-scale MAP corresponds to $\theta = 0.600$, not $0.667$: two analysts fitting the *same model* on different scales report different point estimates.

**What is invariant.**

- **MLE**: invariant by Theorem 4.1 — the likelihood has no Jacobian because it is not a density in $\theta$.
- **Posterior distribution**: invariant as a *measure*; probabilities of corresponding sets are equal.
- **Posterior median and other quantiles**: invariant under monotone $g$ (quantiles commute with monotone maps).
- **Posterior mean**: not invariant either ($E[g(\theta)]\ne g(E[\theta])$), but at least it is defined by an explicit loss.
- **MAP**: invariant only for affine $g$.

$$
\boxed{\hat\theta_{\text{MAP}} = 2/3 \text{ but } \sigma\left(\hat\phi_{\text{MAP}}\right) = 3/5: \text{ the mode depends on the coordinate system}}
$$

*Key takeaway:* Modes are density-dependent and densities carry Jacobians; treat MAP as a regularized optimizer, and quote the full posterior when the parameterization is arbitrary.

The cell below recomputes every number claimed in Problem L3.3.

In [20]:
a, b = 3.0, 2.0                                    # posterior Beta(3, 2)
mode_theta = (a - 1) / (a + b - 2)
th = np.linspace(1e-9, 1 - 1e-9, 4_000_001)
mode_theta_grid = th[np.argmax((a - 1) * np.log(th) + (b - 1) * np.log1p(-th))]
print(f"MAP on the theta scale : formula {mode_theta:.6f}, grid {mode_theta_grid:.6f}  (hand 2/3)")
assert np.isclose(mode_theta, 2 / 3) and np.isclose(mode_theta_grid, 2 / 3, atol=1e-5)

phi = np.linspace(-12, 12, 4_000_001)              # logit scale carries a Jacobian theta(1-theta)
th_of_phi = expit(phi)
logdens_phi = ((a - 1) * np.log(th_of_phi) + (b - 1) * np.log1p(-th_of_phi)
               + np.log(th_of_phi) + np.log1p(-th_of_phi))
phi_mode = phi[np.argmax(logdens_phi)]
print(f"MAP on the logit scale : phi* = {phi_mode:.6f}, sigmoid(phi*) = {expit(phi_mode):.6f}  (hand 3/5)")
print(f"g(theta-MAP) = logit(2/3) = {np.log(2.0):.6f}, but phi-MAP = {phi_mode:.6f}")
assert np.isclose(expit(phi_mode), 0.6, atol=1e-5)
assert not np.isclose(phi_mode, np.log(2.0), atol=1e-3)

print(f"\nposterior median is invariant: F^-1(0.5) on theta = "
      f"{stats.beta.ppf(0.5, a, b):.6f}, mapped = {np.log(stats.beta.ppf(0.5, a, b)/(1-stats.beta.ppf(0.5, a, b))):.6f}")
print(f"logit-scale median of the transformed law = "
      f"{np.log(stats.beta.ppf(0.5, a, b)/(1-stats.beta.ppf(0.5, a, b))):.6f}  (identical)")
print(f"posterior mean = {a/(a+b):.6f} (not the mode, not invariant either)")

MAP on the theta scale : formula 0.666667, grid 0.666667  (hand 2/3)


MAP on the logit scale : phi* = 0.405468, sigmoid(phi*) = 0.600001  (hand 3/5)
g(theta-MAP) = logit(2/3) = 0.693147, but phi-MAP = 0.405468

posterior median is invariant: F^-1(0.5) on theta = 0.614272, mapped = 0.465307
logit-scale median of the transformed law = 0.465307  (identical)
posterior mean = 0.600000 (not the mode, not invariant either)


### Problem L3.4 — Wilks' Theorem — Why Twice the Log-Likelihood Ratio Is $\chi^2$

**Statement**

Let $\theta\in\mathbb{R}^d$ and test $H_0 : \theta \in \Theta_0$ of dimension $d - k$ against $H_1 : \theta\in\Theta$. Show that under $H_0$,

$$
\Lambda_n = 2\left[\ell(\hat\theta) - \ell(\hat\theta_0)\right] \xrightarrow{d} \chi^2_k .
$$

Then apply it to model selection and connect it to the AIC penalty.

**Intuition**

Near its maximum the log-likelihood becomes a paraboloid of curvature $nI$, so a likelihood ratio is a squared distance. Restricting to a submodel is an orthogonal projection, and squared norms of projected standard Gaussians are $\chi^2$.

**Solution**

**Setup.** Assume the simple case $\Theta_0 = \{\theta : \theta_{(2)} = 0\}$ with $\theta = (\theta_{(1)},\theta_{(2)})$, $\theta_{(2)}\in\mathbb{R}^k$, and let $\theta_0$ be the true parameter, interior to $\Theta_0$.

**Step 1 — quadratic expansion.** Taylor-expand $\ell$ about the unconstrained MLE $\hat\theta$, where the score vanishes:

$$
\ell(\theta) \approx \ell(\hat\theta) - \frac{n}{2}\left(\theta - \hat\theta\right)^{\top}\hat{I}\left(\theta-\hat\theta\right), \qquad \hat{I} = -\frac1n\nabla^2\ell(\hat\theta) \xrightarrow{P} I(\theta_0) .
$$

So the log-likelihood is asymptotically an exact quadratic form — the log-likelihood surface becomes a paraboloid whose curvature is the Fisher information.

**Step 2 — the statistic is a squared distance.** Applying the expansion at $\hat\theta_0$ (the constrained maximizer),

$$
\Lambda_n = 2\left[\ell(\hat\theta) - \ell(\hat\theta_0)\right] \approx n\left(\hat\theta_0-\hat\theta\right)^{\top}I\left(\hat\theta_0-\hat\theta\right) ,
$$

which is the squared $I$-weighted distance between the unconstrained and constrained estimates.

**Step 3 — a projected Gaussian.** By Theorem 4.5, $\sqrt{n}\left(\hat\theta - \theta_0\right)\xrightarrow{d} Z \sim \mathcal{N}\left(0, I^{-1}\right)$. Whitening, set $W = I^{1/2}\sqrt{n}(\hat\theta - \theta_0) \xrightarrow{d}\mathcal{N}(0, I_d)$. The constrained MLE is asymptotically the $I$-orthogonal projection of $\hat\theta$ onto the linear space $\Theta_0$, so in whitened coordinates $\sqrt{n}\,I^{1/2}(\hat\theta_0 - \hat\theta) \to -(I_d - P)W$ where $P$ is an orthogonal projection of rank $d-k$. Therefore

$$
\Lambda_n \xrightarrow{d} \left\Vert (I_d - P)W\right\Vert^2 .
$$

**Step 4 — Cochran's theorem.** $(I_d - P)$ is an orthogonal projection of rank $k$, and the squared norm of a standard Gaussian projected onto a $k$-dimensional subspace is $\chi^2_k$. Hence $\Lambda_n \xrightarrow{d}\chi^2_k$. $\blacksquare$

**Consequences.**

- **Testing.** Reject $H_0$ at level $\alpha$ when $\Lambda_n \gt \chi^2_{k,1-\alpha}$. The degrees of freedom are the number of *constrained* parameters, independent of the model's details — the same universality as the CLT, here for likelihood surfaces.
- **Adding a parameter to a null model.** Under $H_0$, $E[\Lambda_n] = k$: each spurious parameter buys $\tfrac{1}{2}$ of log-likelihood on average, purely by chance. This is exactly the optimism that AIC $= 2k - 2\ell(\hat\theta)$ corrects for, and it explains why in-sample likelihood can never be used to compare nested models directly.
- **Regularity is essential.** Wilks fails on boundaries (e.g. testing a variance component $\sigma^2 = 0$ gives a $\tfrac12\chi^2_0 + \tfrac12\chi^2_1$ mixture) and for non-identified nuisance parameters under $H_0$ (testing the number of mixture components).

$$
\boxed{2\left[\ell(\hat\theta)-\ell(\hat\theta_0)\right]\xrightarrow{d}\chi^2_k; \ \ E[\Lambda_n] = k \Rightarrow \text{AIC penalty } k}
$$

*Key takeaway:* Asymptotically the log-likelihood is a paraboloid with curvature $nI$, so likelihood ratios are squared Gaussian distances — and the $\chi^2$ degrees of freedom simply count the constrained directions.

The cell below recomputes every number claimed in Problem L3.4.

In [21]:
# Nested Gaussian means: H0 constrains k of d coordinates to 0. Under H0, Lambda_n ~ chi2_k.
d, k, n, reps = 5, 2, 400, 20_000
theta0 = np.zeros(d)                               # true parameter lies in Theta_0

xb = rng.normal(0.0, 1.0, size=(reps, d, n)).mean(axis=2)
# loglik of N(theta, I): -n/2 * ||xbar - theta||^2 + const, so 2[l(hat) - l(hat0)] = n * sum of
# the squared constrained coordinates.
Lambda = n * np.sum(xb[:, d - k:] ** 2, axis=1)

print(f"E[Lambda_n] = {Lambda.mean():.4f}   predicted k = {k}")
print(f"Var[Lambda_n] = {Lambda.var():.4f}  predicted 2k = {2*k}")
qs = np.array([0.5, 0.75, 0.9, 0.95, 0.99])
print("\nquantiles of Lambda_n vs chi2_k:")
for q, emp in zip(qs, np.quantile(Lambda, qs)):
    print(f"   q = {q:4.2f}   empirical {emp:7.4f}   chi2_{k} {stats.chi2.ppf(q, k):7.4f}")
assert np.isclose(Lambda.mean(), k, rtol=0.05)
assert np.allclose(np.quantile(Lambda, qs), stats.chi2.ppf(qs, k), rtol=0.05)

ks = stats.kstest(Lambda, "chi2", args=(k,))
print(f"\nKolmogorov-Smirnov statistic against chi2_{k}: D = {ks.statistic:.5f}, p = {ks.pvalue:.4f}")
assert ks.statistic < 0.02

rate = np.mean(Lambda > stats.chi2.ppf(0.95, k))   # size of the level-0.05 test
print(f"rejection rate at the 0.95 critical value: {rate:.4f}  (nominal 0.05)")
print(f"AIC reading: E[Lambda_n]/2 = {Lambda.mean()/2:.4f} extra log-likelihood per {k} "
      f"spurious parameters, i.e. 1/2 each")
assert abs(rate - 0.05) < 0.01

E[Lambda_n] = 2.0093   predicted k = 2
Var[Lambda_n] = 3.9971  predicted 2k = 4

quantiles of Lambda_n vs chi2_k:
   q = 0.50   empirical  1.3921   chi2_2  1.3863
   q = 0.75   empirical  2.7879   chi2_2  2.7726
   q = 0.90   empirical  4.6128   chi2_2  4.6052
   q = 0.95   empirical  6.0042   chi2_2  5.9915
   q = 0.99   empirical  9.1376   chi2_2  9.2103

Kolmogorov-Smirnov statistic against chi2_2: D = 0.00702, p = 0.2773
rejection rate at the 0.95 critical value: 0.0503  (nominal 0.05)
AIC reading: E[Lambda_n]/2 = 1.0047 extra log-likelihood per 2 spurious parameters, i.e. 1/2 each
